# Building an Eval for an AI Shopping Assistant

In this build-along, you'll learn how to **build an eval** for an AI agent and use it to systematically find and fix problems.

`boutique` is a simple shopping assistant that looks up product prices and does math. It works... mostly. Your job is to figure out exactly where it breaks, why, and how to fix it.

---

**Prepared for Partner Basecamp participants.** Not for reproduction or redistribution as training material — you're free to apply these patterns in your own client work.

In [1]:
# Install dependencies into THIS kernel — safe to re-run; survives locked-down (PEP 668) Pythons.
import importlib.util, os, subprocess, sys

# ── Environment guard ─────────────────────────────────────────────────────
# Lives in the SAME cell as the installer below so it can't be skipped: no
# package is ever installed into a bare system Python (the old worst case was
# --break-system-packages against an IT-managed machine). If this stops you,
# see SETUP.md → "Why the notebook just stopped".
def _in_isolated_env():
    """True when the RUNNING KERNEL is a venv/virtualenv/conda env or Colab.
    Judged from the interpreter itself (sys.*). Activation env vars inherited
    from the launching shell are trusted only when sys.executable actually
    lives inside the environment they point to — a system-Python kernel
    launched from an activated terminal still inherits VIRTUAL_ENV and must
    NOT pass."""
    if "google.colab" in sys.modules:
        return True  # Colab sandboxes its own disposable runtime
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405 venv (python -m venv); also most conda envs
    if hasattr(sys, "real_prefix"):
        return True  # legacy virtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # hearsay from the shell — the kernel lives elsewhere
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # conda's shared `base` doesn't count as isolated
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  Environment guard bypassed (BASECAMP_ALLOW_SYSTEM_PYTHON=1) — "
          "installing into this Python on purpose.")
elif not _in_isolated_env():
    _HEAD = "✗ System Python detected — stopped before installing anything"
    _BODY = (
        "Installing packages here changes Python for your whole machine — on a\n"
        "corporate-managed laptop that can mean an IT ticket.\n"
        "\n"
        "Fix (2 steps):\n"
        "  1. In a terminal, from the repo root:\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. In VS Code: click the kernel name (top-right) → Select Another Kernel →\n"
        "     Python Environments → pick the one ending in .venv → run this cell again.\n"
        "\n"
        "Using conda? `conda activate <env>` (not `base`), then pick that kernel.\n"
        "Facilitator on a self-managed machine? BASECAMP_ALLOW_SYSTEM_PYTHON=1 bypasses."
    )
    _shown = False
    try:  # same banner treatment as the API-key check below (green box, red variant)
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "Environment guard stopped this cell — see the message above."
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ Environment guard: isolated interpreter detected, safe to install")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: list of (import_name, pip_spec). Install only what is missing,
    into the running interpreter. Tries a normal install, then user-space, then a
    PEP 668 override (user-space first, system-wide only as a last resort). Every
    attempt is silent — pip's output is captured, not streamed — so a locked-down
    Python (Homebrew or Debian, PEP 668) no longer dumps a scary
    'externally-managed-environment' wall of text when a fallback is what actually
    succeeds. Only if every strategy fails does it surface the reason, with the
    venv fix instead of a raw traceback."""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "(no output from pip)"
    raise SystemExit(
        "\n  Couldn't install: " + ", ".join(missing) + "\n"
        "  This Python is locked down (PEP 668) or offline. Quickest fix is a venv:\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows: see SETUP.md\n"
        f"      pip install {' '.join(missing)}\n"
        "  Then pick the .venv interpreter in VS Code (kernel picker, top-right) and Run All.\n"
        "  Corporate proxy or PyPI blocked? See SETUP.md in the repo root.\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ Dependencies ready")

✓ Environment guard: isolated interpreter detected, safe to install
✓ Dependencies ready


### Setup — connect to Claude

Run the next cell first. The setup cell creates a **`.env` file** the first time you run it (gitignored — your key is never committed). Open it, paste your key after `ANTHROPIC_API_KEY=`, save, and re-run — it survives kernel restarts, so you paste once. *(No `.env` yet? A hidden input box appears as a fallback.)* You're locked in when you see the green **"✓ API key verified"** banner. Red banner? Do what it says and run the cell again.

In [2]:
import os

def _status(ok, msg):
    """Green/red banner in notebooks; plain text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Connect to Claude — Anthropic API or Amazon Bedrock ──
# Works with either credential type; the cell figures out which you have.
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  plus  AWS_REGION=us-east-1
# Put whichever you use in the .env file this cell creates (gitignored — never committed),
# or export it in your shell. A value in the shell wins over the .env file.
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """Nearest existing .env walking up from the working dir (so one root .env serves every
    exercise); if none exists yet, point at the repo root — or this folder if the notebook
    was opened on its own."""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# Tiny .env parser (no python-dotenv dependency). Re-read on every run, so pasting your
# key and re-running picks it up. A real value in the environment (shell / Claude Code / CI)
# wins; the placeholder never sticks.
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """Warning-yellow banner + stop, so setup fails here rather than several cells later."""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("Credentials missing — see the message above.")


if PROVIDER is None:
    _needs_credentials(
        "📋 Add your credentials to continue",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Using the Anthropic API? Set:\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Using Amazon Bedrock? Set both:\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # the region your models are enabled in\n"
        "\n"
        "Save the file, then click ▶ on this cell again."
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrock needs a region",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "and add the region your Bedrock models are enabled in, e.g.:\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "Save the file, then click ▶ on this cell again."
    )


def _model(name):
    """Bedrock model IDs carry an `anthropic.` prefix; the Anthropic API uses the bare ID."""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# Named models the exercise uses — resolved for whichever provider you're on.
MODEL = _model("claude-sonnet-5")        # the workhorse for this exercise
FAST_MODEL = _model("claude-haiku-4-5")  # cheap + quick (connection check, judges)
BIG_MODEL = _model("claude-opus-4-8")    # when you want to try a larger model


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# Connection check — verifies the credential AND that this model is reachable for you.
# On Bedrock a valid key can still 404 if the model isn't enabled in your account/region,
# so we ping the real model ID rather than just checking the credential's shape.
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("Model not available — see the message above.")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "That Bedrock key was rejected. Check AWS_BEARER_TOKEN_BEDROCK and that "
                       "it has Bedrock invoke permissions, then run this cell again.")
    else:
        _status(False, "That key was rejected. Run this cell again and paste the whole key "
                       "(it starts with sk-ant-).")
    raise SystemExit("Credentials not accepted - re-run this cell and try again.")
except Exception as exc:
    _status(False, "Could not reach the API (" + type(exc).__name__ + "). Check your "
                   "connection, then run this cell again.")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # later cells / !python pick it up
        _status(True, "API key verified - you're connected to Claude.")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# The working client. Longer timeout: needed for max_tokens>21333 with non-streaming calls.
client = _make_client(timeout=900.0)


## Why Evals?

When you're building with Claude, "try it a few times and see if it works" isn't the best strategy. Vibes are definitely important, but Evals give you:

- **A baseline.** How good is the agent right now? Across what types of queries?
- **A feedback loop.** Change something (prompt, tool design, model), re-run, see if it actually helped and/or if there are any regressions.
- **Confidence.** Before shipping to a customer, you can point to numbers, not just vibes.

Today's build along:

1. **Meet the agent** and begin to notice what works and what doesn't
2. **Define eval tasks** that cover the agent's capabilities and edge cases
3. **Run the eval** and inspect results systematically
4. **Improve the agent** using eval results, then re-run to verify
5. **Add an LLM-as-judge grader** for queries that can't be checked with simple string matching

---

## Part 1: Meet the Agent

`boutique` is a simple single-turn agent. It has two tools:

| Tool | What it does |
|------|-------------|
| `get_product(product)` | Returns the price of an item from the catalog |
| `calculate(op, input1, input2)` | Basic math operations |

The agentic loop is simple:

```
User query → Claude decides what to do → Tool call (if needed) → Tool result → ... → Final answer
```

Skim through the code below. You might need to come back later to inspect:
- The **tool specs** (what Claude sees/knows about each tool)
- The **system prompt**
- The **tool implementations** (what actually happens when a tool is called)

In [3]:
import math
from anthropic import Anthropic
from anthropic.types import ToolUseBlock, TextBlock

# ── Config ────────────────────────────────────────────────────────────────────

MODEL = FAST_MODEL  # Haiku, resolved for your provider in setup (alias — never 404s on retirement)
SYSTEM_PROMPT = "You are a helpful assistant."

# The harness client: SDK defaults (long timeout, standard retries). The verification
# client above uses a short 30s timeout / 1 retry — right for a 1-token ping, too
# twitchy for a full eval run. Re-create it; the key is in the environment now.
client = Anthropic()

# ── Tool implementations ─────────────────────────────────────────────────────

def get_product(product: str):
    catalog = {
        "jeans": 49.99,
        "shirt": 29.99,
        "dress": 59.99,
        "jacket": 89.99,
        "sneakers": 74.99,
        "hat": 19.99,
        "socks": 9.99,
        "hoodie": 44.99,
        "shorts": 34.99,
        "t-shirt": 24.99,
        "sweater": 54.99,
        "belt": 24.99,
    }
    return catalog[product]


def calculate(op: str, input1: float, input2: float):
    if op == "+": return input1 + input2
    elif op == "-": return input1 - input2
    elif op == "*": return input1 * input2
    elif op == "/": return input1 / input2
    elif op == "**": return input1 ** input2

TOOL_REGISTRY = {
    "get_product": get_product,
    "calculate": calculate,
}

# ── Tool specs (sent to Claude) ──────────────────────────────────────────────

GET_PRODUCT_SPEC = {
    "name": "get_product",
    "description": "get_product",
    "input_schema": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "product",
            },
        },
        "required": ["product"],
    },
}

CALCULATE_SPEC = {
    "name": "calculate",
    "description": "calculator",
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "description": "operator",
            },
            "input1": {
                "type": "number",
                "description": "input1",
            },
            "input2": {
                "type": "number",
                "description": "input2",
            },
        },
        "required": ["op", "input1", "input2"],
    },
}

ALL_TOOL_SPECS = [GET_PRODUCT_SPEC, CALCULATE_SPEC]

# ── Agent ─────────────────────────────────────────────────────────────────────

def call_claude(messages, tools, model=None):
    return client.messages.create(
        model=model or MODEL,
        system=SYSTEM_PROMPT,
        max_tokens = 1024,
        tools=tools,
        messages=messages,
    )


def execute_tool(name, inputs):
    try:
        return str(TOOL_REGISTRY[name](**inputs))
    except Exception as e:
        return f"Error: {e}"


def run_agent(prompt, eval_mode=False, model=None):
    messages = [{"role": "user", "content": prompt}]
    total_input_tokens = 0
    total_output_tokens = 0

    while True:
        response = call_claude(messages, tools=ALL_TOOL_SPECS, model=model)
        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens
        messages.append({"role": "assistant", "content": response.content})

        # Break unless Claude asked for a tool. Guarding on "tool_use" (rather than
        # "end_turn") prevents a looping 400 if the model stops for another reason
        # (e.g. max_tokens) — we'd otherwise send back an empty tool_results message.
        # (With server-side tools, also handle stop_reason == "pause_turn".)
        if response.stop_reason != "tool_use":
            break

        tool_calls = [block for block in response.content if isinstance(block, ToolUseBlock)]

        tool_results = []
        for tool_call in tool_calls:
            result = execute_tool(tool_call.name, tool_call.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_call.id,
                "content": result,
            })

        messages.append({"role": "user", "content": tool_results})

    if eval_mode:
        return {
            "messages": messages,
            "usage": {"input_tokens": total_input_tokens, "output_tokens": total_output_tokens},
        }

    return "\n".join(block.text for block in response.content if isinstance(block, TextBlock))


print("boutique agent ready.")

boutique agent ready.


### Try It Out

The cell below starts an interactive session with the agent automatically. Type queries and see how it responds. Type `quit` to stop. (Running the whole notebook with Run All? Set `INTERACTIVE_CHAT = False` in that cell first, or just type `quit` when it pauses here.)

Some queries to try:
- `How much do jeans cost?` (simple, should work)
- `Price of a t-shirt?` (will the hyphen trip it up?)
- `How much for shoes?` ("shoes" isn't in the catalog, but "sneakers" is)
- `3 shirts and 2 belts, what's my total?` (multi-tool: lookups + math)
- `What's 20% off a jacket?` (requires percentage math)
- `What do you sell?` (can it describe its own capabilities?)

In [4]:
ANTHROPIC_ORANGE = "#E07A5F"  # brand accent — keeps the boutique agent from blending into the notebook's black-on-white output

def _display_chat_hint():
    """Tells you how to turn on the interactive chat: Anthropic-orange in a notebook, plain
    text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        display(HTML(
            f'<div style="background:{ANTHROPIC_ORANGE};color:#fff;padding:10px 16px;'
            f'border-radius:8px;font-family:sans-serif;font-size:14px;">'
            f'🛍️ <b>The Boutique Agent</b> is ready — set '
            f'<code style="background:rgba(255,255,255,.3);padding:1px 5px;border-radius:4px;">INTERACTIVE_CHAT = False</code> '
            f'above and run this cell again to chat.</div>'
        ))
    except Exception:
        print("The Boutique Agent is ready — set INTERACTIVE_CHAT = False above and run this cell again to chat.")


INTERACTIVE_CHAT = False  # chat with the agent by default — set False (and Run All) to skip straight to the eval sections

if not INTERACTIVE_CHAT:
    _display_chat_hint()
else:
    print("Boutique Agent Response Results")

# The input() prompt below is VS Code's own editor UI (like the command palette) — its
# colors always follow the user's VS Code theme and can't be styled from here. Since text
# is the only lever we have, the prompt itself carries the branding instead.
while INTERACTIVE_CHAT:
    query = input("\n🛍️  The Boutique Shopping agent is running, ask your question and press ENTER, press ESC to stop the agent.")
    if not query.strip() or query.strip().lower() in ("quit", "exit", "q"):
        print("Session ended.")
        break
    print(f"\nBoutique: {run_agent(query)}")

---

## Part 2: The Eval Framework

An eval has three main components:

```
Tasks ──> Runner ──> Graders ──> Results
```

- **Tasks** define *what* to test: scenario, expected behavior, and how to grade it
- **Runner** orchestrates execution: sends scenarios' queries to the agent, collects transcripts, applies graders
- **Graders** check *whether* the agent did the right thing and return a score + reason

The runner and graders are defined in the next two cells. Run them both. You can quickly read through the code to understand how they work, but you won't need to modify them (yet).

### Available graders

| Grader | What it checks | Check format |
|--------|---------------|---------------|
| `response_contains` | Final text contains a string (case-insensitive) | `"49.99"` or `"jeans"` |
| `response_numeric` | Final text contains a number within tolerance | `{"value": 49.99, "tolerance": 0.05}` |
| `tool_use` | Agent called a specific tool with specific args | `{"tool_name": "get_product", "arguments": {"product": "jeans"}}` |

Each grader returns a binary score (0 = fail, 1 = pass) and a reason explaining why. A task passes only if **all checks from all graders** pass.

In [5]:
# ── Graders (just run this cell) ──────────────────────────────────────────────

import re

def grade_response_contains(result, check, context=None):
    text = result["final_text"].lower()
    target = check.lower()
    if target in text:
        return {"score": 1.0, "reason": f"Found '{check}' in response"}
    return {"score": 0.0, "reason": f"'{check}' not found in response: {result['final_text'][:200]}"}


def grade_response_numeric(result, check, context=None):
    if isinstance(check, (int, float)):
        value, tolerance = float(check), 0.01
    else:
        value = float(check["value"])
        tolerance = float(check.get("tolerance", 0.01))

    numbers = re.findall(r"-?[\d,]+\.?\d*", result["final_text"])
    for num_str in numbers:
        try:
            num = float(num_str.replace(",", ""))
            if abs(num - value) <= tolerance:
                return {"score": 1.0, "reason": f"Found {num} (expected {value} +/- {tolerance})"}
        except ValueError:
            continue
    return {"score": 0.0, "reason": f"Expected {value} (+/- {tolerance}), found: {numbers[:10]}"}


def grade_tool_use(result, check, context=None):
    tool_name = check["tool_name"]
    expected_args = check.get("arguments", None)

    for call in result["tool_calls"]:
        if call["name"] != tool_name:
            continue
        if expected_args is None:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' was called"}

        # Partial match: only check specified keys
        actual_args = call.get("arguments", {})
        match = all(
            (isinstance(v, str) and isinstance(actual_args.get(k), str) and v.lower() == actual_args[k].lower())
            or actual_args.get(k) == v
            for k, v in expected_args.items()
        )
        if match:
            return {"score": 1.0, "reason": f"Tool '{tool_name}' called with matching args: {expected_args}"}

    actual = [{"name": c["name"], "args": c.get("arguments", {})} for c in result["tool_calls"]]
    if expected_args:
        return {"score": 0.0, "reason": f"'{tool_name}' not called with {expected_args}. Actual: {actual}"}
    return {"score": 0.0, "reason": f"'{tool_name}' never called. Actual: {[c['name'] for c in result['tool_calls']]}"}


GRADER_REGISTRY = {
    "response_contains": grade_response_contains,
    "response_numeric": grade_response_numeric,
    "tool_use": grade_tool_use,
}

print(f"Graders loaded: {list(GRADER_REGISTRY.keys())}")

Graders loaded: ['response_contains', 'response_numeric', 'tool_use']


In [6]:
# ── Eval Runner (just run this cell) ──────────────────────────────────────────

import json, os, time, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed


def parse_transcript(messages):
    """Extract final_text and tool_calls from raw agent transcript."""
    final_text, tool_calls = "", []
    for msg in messages:
        if msg["role"] != "assistant":
            continue
        for block in msg["content"]:
            if isinstance(block, TextBlock):
                final_text = block.text
            elif isinstance(block, ToolUseBlock):
                tool_calls.append({"name": block.name, "arguments": block.input, "id": block.id})
    # Match tool results back to calls
    for msg in messages:
        if msg["role"] != "user" or not isinstance(msg["content"], list):
            continue
        for item in msg["content"]:
            if isinstance(item, dict) and item.get("type") == "tool_result":
                for call in tool_calls:
                    if call["id"] == item["tool_use_id"]:
                        call["result"] = item.get("content", "")
                        break
    return {"final_text": final_text, "tool_calls": tool_calls, "messages": messages}


def run_single_task(agent_fn, task, model=None):
    """Run one task, apply graders, return result with grades + metrics."""
    start = time.time()
    try:
        raw = agent_fn(task["query"], eval_mode=True, model=model)
    except Exception:
        return {
            "task_id": task["id"], "task_description": task.get("description", ""),
            "query": task["query"], "category": task.get("category", ""),
            "error": traceback.format_exc(), "passed": False, "grades": [],
            "metrics": {"time": time.time() - start},
        }

    elapsed = time.time() - start
    result = parse_transcript(raw["messages"])
    usage = raw.get("usage", {})
    turns = sum(1 for m in raw["messages"] if m["role"] == "assistant")
    metrics = {
        "time": round(elapsed, 3), "tool_calls": len(result["tool_calls"]),
        "turns": turns, "input_tokens": usage.get("input_tokens", 0),
        "output_tokens": usage.get("output_tokens", 0),
    }

    grades = []
    context = {"query": task["query"], "task_id": task["id"], "model": model}
    for grader in task.get("graders", []):
        grader_fn = GRADER_REGISTRY.get(grader["type"])
        if grader_fn is None:
            grades.append({"type": grader["type"], "check": None, "score": 0.0, "reason": f"Unknown grader: {grader['type']}"})
            continue
        for check in grader.get("checks", []):
            # A grader that raises (or returns something malformed) fails this one
            # task only — without the guard, the exception would re-raise at
            # f.result() in run_eval and abort the entire run.
            try:
                grade = grader_fn(result, check, context)
                grades.append({"type": grader["type"], "check": check, "score": grade["score"], "reason": grade["reason"]})
            except Exception as exc:
                grades.append({"type": grader["type"], "check": check, "score": 0.0,
                               "reason": f"grader error: {type(exc).__name__}: {exc}"})

    passed = all(g["score"] == 1.0 for g in grades) if grades else False

    return {
        "task_id": task["id"], "task_description": task.get("description", ""),
        "query": task["query"], "category": task.get("category", ""),
        "passed": passed, "grades": grades, "metrics": metrics,
        "final_text": result["final_text"],
        "transcript": [
            block.model_dump() if hasattr(block, "model_dump") else block
            for msg in raw["messages"]
            for block in (msg["content"] if isinstance(msg["content"], list) else [msg["content"]])
        ],
    }


def run_eval(agent_fn, tasks, model=None, num_runs=1, max_workers=5):
    """Run the full eval suite. Returns structured results."""
    all_runs = []
    for _ in range(num_runs):
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(run_single_task, agent_fn, t, model): t for t in tasks}
            run_results = []
            for f in as_completed(futures):
                r = f.result()
                run_results.append(r)
                mark = "PASS" if r["passed"] else ("ERROR" if r.get("error") else "FAIL")
                print(f"  [{len(run_results)}/{len(tasks)}] {r['task_id']}: {mark}", flush=True)
        task_order = {t["id"]: i for i, t in enumerate(tasks)}
        run_results.sort(key=lambda r: task_order.get(r["task_id"], 999))
        all_runs.append(run_results)
    return {"runs": all_runs, "config": {"model": model, "num_runs": num_runs, "num_tasks": len(tasks)}}


def save_results(results, directory="eval_results"):
    """Save eval results to a JSON file."""
    os.makedirs(directory, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    model_name = results["config"].get("model") or "default"
    model_short = model_name.split("-")[1] if "-" in str(model_name) else model_name
    filename = f"{directory}/eval_{model_short}_{timestamp}.json"
    with open(filename, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Results saved to {filename}")
    return filename


def print_summary(results):
    """Print formatted eval results."""
    config = results["config"]
    print(f"{'=' * 60}")
    print(f"EVAL RESULTS: {config['num_tasks']} tasks, {config['num_runs']} run(s)")
    if config.get("model"): print(f"Model: {config['model']}")
    print(f"{'=' * 60}\n")

    for run_idx, run in enumerate(results["runs"]):
        if config["num_runs"] > 1: print(f"--- Run {run_idx + 1} ---")
        passed = sum(1 for r in run if r["passed"])
        total = len(run)
        print(f"Overall: {passed}/{total} passed ({passed/total*100:.0f}%)\n")

        # Per-category breakdown
        categories = {}
        for r in run:
            cat = r.get("category", "uncategorized")
            categories.setdefault(cat, {"passed": 0, "total": 0})
            categories[cat]["total"] += 1
            if r["passed"]: categories[cat]["passed"] += 1
        if len(categories) > 1:
            print("By category:")
            for cat, c in sorted(categories.items()):
                print(f"  {cat}: {c['passed']}/{c['total']} ({c['passed']/c['total']*100:.0f}%)")
            print()

        # Per-task detail
        print("Tasks:")
        for r in run:
            mark = "PASS" if r["passed"] else "FAIL"
            print(f"  [{mark}] {r['task_id']}: {r['task_description']}")
            for g in r.get("grades", []):
                print(f"    {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason'][:120]}")
            if r.get("error"): print(f"    Error: {r['error'][:200]}")

        # Aggregate metrics
        ok = [r for r in run if not r.get("error")]
        if ok:
            print(f"\nMetrics (avg): {sum(r['metrics']['time'] for r in ok)/len(ok):.2f}s, "
                  f"{sum(r['metrics']['tool_calls'] for r in ok)/len(ok):.1f} tool calls, "
                  f"{sum(r['metrics']['turns'] for r in ok)/len(ok):.1f} turns")
            print(f"Tokens: {sum(r['metrics']['input_tokens'] for r in ok):,} in, "
                  f"{sum(r['metrics']['output_tokens'] for r in ok):,} out")
        print()


def inspect_task(results, task_id, run_index=0):
    """Print detailed results for a specific task including transcript."""
    run = results["runs"][run_index]
    r = next((r for r in run if r["task_id"] == task_id), None)
    if r is None:
        print(f"Task '{task_id}' not found"); return

    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['task_id']}: {r['task_description']}")
    print(f"Query: {r['query']}")
    print(f"Response: {r.get('final_text', 'N/A')}\n")
    if r.get("error"): print(f"ERROR:\n{r['error']}"); return

    print("Grades:")
    for g in r["grades"]:
        print(f"  {'+' if g['score'] == 1.0 else '-'} {g['type']}: {g['reason']}")
    print(f"\nMetrics: {r['metrics']}")

    print("\nTranscript:")
    for item in r.get("transcript", []):
        if isinstance(item, dict):
            t = item.get("type", "?")
            if t == "text": print(f"  [text] {item.get('text', '')[:300]}")
            elif t == "tool_use": print(f"  [tool_use] {item.get('name', '?')}({item.get('input', {})})")
            elif t == "tool_result": print(f"  [tool_result] {str(item.get('content', ''))[:200]}")
            else: print(f"  [{t}] {str(item)[:200]}")
        else: print(f"  {str(item)[:200]}")


print("Eval framework ready.")

Eval framework ready.


### Design decisions worth noting

If you read through the grader and runner code above, a few choices stand out:

- **Registry dict, not if/elif.** Graders are looked up by type name in `GRADER_REGISTRY`, a plain dict. Adding a new grader type is one function + one dict entry, no need to touch a dispatch chain. This is the pattern most eval frameworks use.

- **The runner doesn't print.** `run_eval()` returns a data structure; `print_summary()` is a separate function. This keeps the runner testable and reusable. You could swap in a different display, save to JSON, or feed results into a dashboard without changing the runner.

- **Agent as parameter.** The runner takes `agent_fn` as an argument instead of importing the agent directly. This makes it easy to test with a mock, swap in a different agent, or wrap the agent with instrumentation.

- **Runner owns transcript parsing.** The agent returns raw messages; the runner extracts `final_text`, `tool_calls`, etc. This avoids duplicate extraction logic and keeps the agent clean. It doesn't need to know how it will be evaluated.

- **Failed runs ≠ failed tasks.** If the agent throws an exception (API error, timeout), that's a *failed run*, captured in the `error` field. Grading is skipped. If the agent returns the wrong answer, that's a *failed task* : graders run normally and report what went wrong. The distinction matters. Errors mean the infrastructure broke; failures mean the agent gave the wrong answer.

- **Concurrency.** Tasks run in parallel using `ThreadPoolExecutor`, the standard pattern for I/O-bound API calls. Unbounded parallelism would hit rate limits. If your agent used the async `AsyncAnthropic` client instead, you'd replace this with `asyncio.gather()`.

---

### 🧭 Decision log 1 — read the agent before writing a single task

I worked this exercise with [**Jackdaws**](https://github.com/Hazeley-Consulting/jackdaws), my Claude Skill toolkit, as the discipline layer. Three of its skills carry this notebook:

| Jackdaws skill | Role here |
|---|---|
| `eval-and-testing-best-practices` | The *teaching* layer — grader taxonomy, capability vs regression, pass@k vs pass^k, judge bias |
| `prompt-evaluation-best-practices` | The *verification* layer — rules PI001–PI014 and `prompt_eval_audit.py`, which I run against this notebook's own harness in Part 7 |
| `agents-best-practices` | Tool-spec and error-format guidance for the agent fixes in Part 5b |

Every decision cell below names the rule or reference behind the call, so you can disagree with a specific one rather than with the whole thing.

### Why read the code first

It's tempting to jump straight to writing tasks from the six suggested queries. I read cell 6 line by line first, because **a task suite that doesn't hit the actual defects measures nothing**. Here is what's broken, and it's more than the exercise hints at:

| # | Defect | Where | Consequence |
|---|---|---|---|
| **D1** | `catalog[product]` — bare dict index, no normalisation | `get_product` | `"shoes"`, `"T-shirt"`, `" jeans"` all raise `KeyError`. The agent sees `Error: 'shoes'` — no hint that the tool worked and the *item* is the problem, no list of what is stocked |
| **D2** | Descriptions are the tool's own name: `"description": "get_product"`, `"product": {"description": "product"}` | `GET_PRODUCT_SPEC` | Claude is told nothing about what the shop sells or what argument format is valid. It has to guess `"t-shirt"` vs `"tshirt"` vs `"t shirt"` |
| **D3** | `calculate` falls off the end of the `if/elif` chain for an unknown `op` and returns `None` | `calculate` | `str(None)` → the string `"None"` flows back as a *successful* tool result. A silent wrong answer, which is worse than an error |
| **D4** | `SYSTEM_PROMPT = "You are a helpful assistant."` | cell 6 | No role, no scope, no instruction to ground prices in tool calls rather than memory |
| **D5** | `MODEL = FAST_MODEL` overwrites the Sonnet id set during setup | cell 6 | The agent under test runs on **Haiku**. Relevant later: it means an LLM judge on `FAST_MODEL` would be grading its own family |
| **D6** | No turn cap, and `max_tokens=1024` truncation is invisible to the runner | `run_agent` | A truncated answer is scored as a *wrong answer* rather than a *failed run* — the exact confusion cell 12 warns about |

### What that changes about the task suite

Two things:

1. **The five suggested queries map almost one-to-one onto D1–D4.** `"Price of a t-shirt?"` probes D2 (does Claude guess the right key?). `"How much for shoes?"` probes D1 (what does a lookup miss feel like?). `"What's 20% off a jacket?"` probes D3 (there is no percent operator, so it must compose). That's a well-designed exercise, and I kept all five verbatim rather than rewriting them.
2. **D5 and D6 aren't covered by any of them**, so they show up later instead — D5 in the judge-model choice (Part 6), D6 in the harness hardening (Part 5b).

---

## Part 3: Define Your Eval Tasks

### Task schema

Each task is a Python dict with these fields:

```python
{
    "id": "unique_task_id",             # Short, descriptive identifier
    "description": "What this tests",    # Human-readable description
    "query": "The user's question",      # What gets sent to the agent
    "category": "product_lookup",        # For grouping results
    "graders": [                          # List of grader declarations
        {
            "type": "response_contains",  # Which grader to use
            "checks": ["49.99"],          # What to check (one or more)
        },
    ],
}
```

### Worked example

Here's a task that checks whether the agent can look up the price of jeans:

```python
{
    "id": "price_jeans",
    "description": "Direct price lookup for jeans",
    "query": "How much do jeans cost?",
    "category": "product_lookup",
    "graders": [
        {"type": "response_contains", "checks": ["49.99"]},
        {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
    ],
}
```

This task uses two graders:
1. `response_contains` checks that the final answer mentions "49.99"
2. `tool_use` checks that the agent actually called `get_product` with `product="jeans"` (didn't hallucinate the price)

### A note on `tool_use` grader brittleness

Tool use checks should verify that the agent *grounded its answer in tools* (didn't hallucinate a price or do mental math), not enforce a rigid call sequence. If the agent reaches the correct answer via a valid but unexpected path, that's fine.

Checking `{"tool_name": "get_product"}` (no arguments) verifies the agent used the tool at all. Checking `{"tool_name": "get_product", "arguments": {"product": "jeans"}}` verifies the exact argument. Only do this when the argument value is the thing you're testing (e.g., synonym resolution).

### Your turn

The jeans task is given as a reference. Now build tasks for these five example queries:

1. `"Price of a t-shirt?"` - Will the agent handle the hyphen correctly? What argument should it pass to `get_product`?
2. `"How much for shoes?"` - "shoes" isn't in the catalog. What *should* happen here, and how do you grade it?
3. `"3 shirts and 2 belts, what's my total?"` - Multiple product lookups plus a calculation. What's the expected total?
4. `"What's 20% off a jacket?"` - Needs both a product lookup and a calculation. What number should you check for?
5. `"What do you sell?"` - Open-ended. Can you grade this with the graders you have? (Hint: come back to this one after Part 6.)

For each query, think about:
- Which category does it belong to?
- Which graders make sense? What checks?
- What's the expected correct answer?
- Is a `tool_use` check needed, or is checking the response enough?

### ✏️ YOUR TURN — write your eval tasks in THIS cell

Each task is a dict that needs an **id**, the **query** (the prompt sent to the agent), and **graders** with **checks** that decide pass/fail — keep the worked example, copy the commented template for each new task, and add yours below the `# ✏️ ADD YOUR TASKS BELOW` marker.

In [7]:
# ✏️ YOUR TURN — write your eval tasks in THIS list.
# Each task needs an id, the query (prompt) to send to the agent, and graders with checks.
#
# Two fields here are mine, not the exercise's:
#   "suite"  — "capability" (behaviour under development, read as pass@k) or
#              "regression" (behaviour that must hold every time, read as pass^k).
#              The runner ignores unknown keys; the metrics cell after Part 4 uses it.
#   comments — the expected value and *why* that value, so a reviewer can audit the
#              answer key without recomputing it from the catalog.
tasks = [
    # ── Reference task (worked example) ─────────────────────────────────────
    {
        "id": "price_jeans",
        "description": "Direct price lookup for jeans",
        "query": "How much do jeans cost?",
        "category": "product_lookup",
        "suite": "regression",  # the simplest thing the agent does — must never break
        "graders": [
            {"type": "response_contains", "checks": ["49.99"]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "jeans"}}]},
        ],
    },

    # ── Build tasks for these queries ──────────────────────────────────────

    # 1. "Price of a t-shirt?"
    #    Catalog key is "t-shirt" (hyphenated). Expected: 24.99.
    #    The argument IS the thing under test here — with D2 (empty tool descriptions)
    #    Claude has to guess between "t-shirt" / "tshirt" / "t shirt", and only one of
    #    those is a key in the dict. So this task keeps the exact-argument check.
    #    A retry path still passes: the grader scans every tool call, so
    #    guess-wrong → KeyError → retry-right is graded as a pass, which is correct —
    #    the agent recovered and the user got the right price.
    {
        "id": "price_tshirt",
        "description": "Hyphenated catalog key — does the agent pass the right argument?",
        "query": "Price of a t-shirt?",
        "category": "product_lookup",
        "suite": "capability",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 24.99, "tolerance": 0.005}]},
            {"type": "tool_use", "checks": [{"tool_name": "get_product", "arguments": {"product": "t-shirt"}}]},
        ],
    },

    # 2. "How much for shoes?"
    #    Deliberately NOT here — it moves to JUDGED_TASKS below. See decision log 2:
    #    "handled the miss gracefully" has no deterministic check, and the obvious
    #    mechanical guard (assert 74.99 is absent) would fail the *desired* answer,
    #    because offering sneakers as the nearest match is exactly what we want.

    # 3. "3 shirts and 2 belts, what's my total?"
    #    shirt 29.99 x 3 = 89.97 ; belt 24.99 x 2 = 49.98 ; total = 139.95.
    #    Graded on the outcome, not the path: any arithmetic route to 139.95 is fine,
    #    so the tool checks only assert that a lookup happened and that the maths went
    #    through `calculate` rather than the model's head.
    {
        "id": "multi_item_total",
        "description": "Two lookups plus multi-step arithmetic",
        "query": "3 shirts and 2 belts, what's my total?",
        "category": "multi_step",
        "suite": "capability",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 139.95, "tolerance": 0.01}]},
            {"type": "tool_use", "checks": [
                {"tool_name": "get_product"},   # grounded in the catalog, not recalled
                {"tool_name": "calculate"},     # arithmetic delegated, not done mentally
            ]},
        ],
    },

    # 4. "What's 20% off a jacket?"
    #    jacket 89.99 x 0.8 = 71.992. Tolerance 0.02 accepts both 71.99 and 71.992,
    #    so rounding style isn't what decides the grade.
    #    There is no percent operator in `calculate` (D3), so the agent has to compose
    #    the discount out of the four it has. That composition is the point of the task.
    {
        "id": "percent_off_jacket",
        "description": "Percentage discount composed from primitive operators",
        "query": "What's 20% off a jacket?",
        "category": "multi_step",
        "suite": "capability",
        "graders": [
            {"type": "response_numeric", "checks": [{"value": 71.99, "tolerance": 0.02}]},
            {"type": "tool_use", "checks": [
                {"tool_name": "get_product", "arguments": {"product": "jacket"}},
                {"tool_name": "calculate"},
            ]},
        ],
    },

    # 5. "What do you sell?"
    #    Also moves to JUDGED_TASKS — the notebook's own hint says to come back to it
    #    after Part 6, and it's the cleanest example of a query with many correct answers.
]

### 🧭 Decision log 2 — how the tasks were graded, and why the corpus gets hashed

**Mechanical first, judgment only when nothing deterministic works.** Jackdaws' `grader-taxonomy.md` puts it bluntly: reach for a deterministic assertion first, and escalate to an LLM judge only when no mechanical check can capture the property, because judgment graders are slower, more expensive and stochastic. Four of the seven tasks came out fully mechanical. The three that didn't are the interesting ones:

| Task | Why it escalates to a judge |
|---|---|
| `unknown_product_shoes` | "Recovered gracefully" has no string form. And the obvious mechanical guard — assert `74.99` is absent — would fail the *best* answer, because "we don't stock shoes, but sneakers are 74.99" is exactly what we want |
| `catalog_overview` | Many correct answers. Any check on specific wording measures phrasing, not correctness |
| `offtopic_redirect` | Half of it *is* mechanical (`tool_call_count` max 0). Only the "didn't fabricate inventory" half needs a judge |

That last row is the pattern worth stealing: **split the task, don't split the difference**. The deterministic half stays deterministic and cheap; the judge is asked one narrow question it's actually good at.

**Argument checks only where the argument is the test.** `tool_use` can check that a tool was called, or that it was called with specific arguments. The second is much more brittle — it can fail an agent that reached the right answer by a valid but unexpected route. So:

- `price_tshirt` pins `{"product": "t-shirt"}`, because the hyphenated key *is* the thing under test (D2 means Claude has to guess it).
- `multi_item_total` pins nothing. It asserts only that `get_product` and `calculate` were both called — grounded in the catalog, arithmetic not done in the model's head — and lets `response_numeric` decide correctness. Any route to 139.95 counts.

This is "grade the outcome, not the path", which the exercise text and Jackdaws' `vocabulary-and-outcomes.md` both land on independently.

**Capability vs regression.** Every task carries a `suite` tag. It's not decoration — it decides which metric the task is read with after Part 4. `price_jeans` and the three judged tasks that encode a *must never* property (don't invent a price, don't fabricate inventory) are regression: they have to hold on every trial. The rest are capability: behaviour still under development, where "can it do this at all" is the useful signal.

**Freezing the corpus (PI001/PI013).** The next cell hashes the suite before anything is measured. The failure this prevents is unglamorous and extremely common: you run a baseline, then think of two more test cases, then run the improved version — and the two scores now describe different test sets. The Jackdaws module that owns this rule opens with a worked case where exactly that happened and a reported "7.33 → 8.17 improvement" turned out to be 7.33 → 6.33 once the confounds were backed out.

The hash covers ids, queries, suite tags and grader definitions — everything that can change a grade. It deliberately excludes `description` and `category`, so improving a human-readable label doesn't read as corpus drift.

In [8]:
# ── The frozen corpus ─────────────────────────────────────────────────────────
# Three things happen here, once, before any number is measured:
#   1. one more mechanical grader (`tool_call_count`)
#   2. the judged tasks (their grader arrives in Part 6 — the criteria are frozen now)
#   3. a sha256 over the whole suite, so a later run can prove it scored the same corpus
#
# Jackdaws PI001/PI013: an eval corpus that is regenerated or extended between runs makes
# two scores describe different test sets. Freezing it here means every number below —
# baseline, ablation rungs, final — is scored against the same 7 tasks.

import hashlib
import json as _json


# ── A fourth mechanical grader ────────────────────────────────────────────────
# Adding one is a function plus a dict entry, exactly as cell 12 promised.
# `{"max": 0}` is how "did not touch the tools at all" becomes a deterministic check;
# `{"max": 6}` doubles as the efficiency grader the Extensions section suggests.
def grade_tool_call_count(result, check, context=None):
    """Count tool calls, optionally for one named tool, against min/max/exact bounds."""
    name = check.get("tool_name")
    calls = [c for c in result["tool_calls"] if name is None or c["name"] == name]
    n = len(calls)
    label = f"'{name}' calls" if name else "tool calls"

    for bound, ok, text in (
        ("exact", lambda v: n == v, "exactly"),
        ("max",   lambda v: n <= v, "at most"),
        ("min",   lambda v: n >= v, "at least"),
    ):
        if bound in check and not ok(check[bound]):
            actual = [c["name"] for c in result["tool_calls"]]
            return {"score": 0.0,
                    "reason": f"expected {text} {check[bound]} {label}, got {n}: {actual}"}

    if not any(b in check for b in ("exact", "max", "min")):
        return {"score": 0.0, "reason": f"tool_call_count check needs one of exact/max/min, got {check}"}
    return {"score": 1.0, "reason": f"{n} {label} satisfies {check}"}


GRADER_REGISTRY["tool_call_count"] = grade_tool_call_count


# ── Tasks that need a judgment grader ─────────────────────────────────────────
# Each `llm_judge` check is ONE criterion. Splitting them means a failure names the
# thing that failed, instead of a single verdict over a bundle of criteria.
JUDGED_TASKS = [
    # Exercise query 2. "shoes" is not a catalog key; "sneakers" is.
    # The regression property is negative — it must never invent a price — and the
    # positive half (offer the nearest real item, or say plainly that we don't stock it)
    # has too many valid phrasings for a string match. The one mechanical check that
    # survives is grounding: it must have actually consulted the catalog.
    {
        "id": "unknown_product_shoes",
        "description": "Catalog miss — recovers without inventing a price",
        "query": "How much for shoes?",
        "category": "graceful_degradation",
        "suite": "regression",
        "graders": [
            {"type": "tool_use", "checks": [{"tool_name": "get_product"}]},
            {"type": "llm_judge", "checks": [
                "The response does not state a specific price for an item called 'shoes' as though that item were in stock.",
                "The response either says the item is unavailable, or offers a specific alternative product from the shop instead.",
            ]},
        ],
    },
    # Exercise query 5. Open-ended by construction.
    {
        "id": "catalog_overview",
        "description": "Describes what the shop actually stocks",
        "query": "What do you sell?",
        "category": "capabilities",
        "suite": "capability",
        "graders": [
            {"type": "llm_judge", "checks": [
                "The response names at least three specific clothing or accessory items that the shop sells.",
                "The response does not claim to sell product categories outside clothing and accessories (for example food, electronics or furniture).",
            ]},
        ],
    },
    # From the Extensions list: an off-topic query. The mechanical half is strong here —
    # a shopping assistant has no reason to call a price lookup for this — so the judge
    # only has to cover the tone/redirect half.
    {
        "id": "offtopic_redirect",
        "description": "Out-of-scope question — answers without reaching for tools",
        "query": "What's the meaning of life?",
        "category": "out_of_scope",
        "suite": "regression",
        "graders": [
            {"type": "tool_call_count", "checks": [{"max": 0}]},
            {"type": "llm_judge", "checks": [
                "The response does not fabricate any product price or shop inventory detail.",
            ]},
        ],
    },
]

FULL_SUITE = tasks + JUDGED_TASKS


def corpus_fingerprint(suite):
    """sha256 over the graded content of a suite: ids, queries and grader definitions.

    Descriptions and categories are deliberately excluded — editing a human-readable
    label should not read as 'you are now scoring a different corpus'. Anything that
    can change a grade is inside the hash.
    """
    canonical = [
        {"id": t["id"], "query": t["query"], "suite": t.get("suite"), "graders": t.get("graders", [])}
        for t in sorted(suite, key=lambda t: t["id"])
    ]
    blob = _json.dumps(canonical, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(blob.encode()).hexdigest()[:12]


CORPUS_SHA = corpus_fingerprint(FULL_SUITE)

# k, and the concurrency the runs are measured at. Both are held constant across every
# measured run below — changing either between two runs makes them incomparable.
K = 5
MAX_WORKERS = 5

print(f"Frozen corpus: {len(FULL_SUITE)} tasks  (sha {CORPUS_SHA})")
for _t in FULL_SUITE:
    _kinds = ", ".join(sorted({g["type"] for g in _t["graders"]}))
    print(f"  {_t['id']:<24} {_t.get('suite',''):<11} [{_kinds}]")
print(f"\nk = {K} trials per task, max_workers = {MAX_WORKERS} — held constant for every measured run.")
print(f"Graders available: {sorted(GRADER_REGISTRY)}")

Frozen corpus: 7 tasks  (sha f4af23d76593)
  price_jeans              regression  [response_contains, tool_use]
  price_tshirt             capability  [response_numeric, tool_use]
  multi_item_total         capability  [response_numeric, tool_use]
  percent_off_jacket       capability  [response_numeric, tool_use]
  unknown_product_shoes    regression  [llm_judge, tool_use]
  catalog_overview         capability  [llm_judge]
  offtopic_redirect        regression  [llm_judge, tool_call_count]

k = 5 trials per task, max_workers = 5 — held constant for every measured run.
Graders available: ['response_contains', 'response_numeric', 'tool_call_count', 'tool_use']


---

## Part 4: Run the Eval

Now let's run the eval and see how the agent performs. Results are also saved as JSON files so you can compare across runs.

In [9]:
results = run_eval(run_agent, tasks)
print_summary(results)
save_results(results)

  [1/4] multi_item_total: FAIL


  [2/4] percent_off_jacket: FAIL


  [3/4] price_jeans: PASS


  [4/4] price_tshirt: PASS


EVAL RESULTS: 4 tasks, 1 run(s)

Overall: 2/4 passed (50%)

By category:
  multi_step: 0/2 (0%)
  product_lookup: 2/2 (100%)

Tasks:
  [PASS] price_jeans: Direct price lookup for jeans
    + response_contains: Found '49.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}
  [PASS] price_tshirt: Hyphenated catalog key — does the agent pass the right argument?
    + response_numeric: Found 24.99 (expected 24.99 +/- 0.005)
    + tool_use: Tool 'get_product' called with matching args: {'product': 't-shirt'}
  [FAIL] multi_item_total: Two lookups plus multi-step arithmetic
    - response_numeric: Expected 139.95 (+/- 0.01), found: [',', '1.', '2.', ',', '3', '2']
    - tool_use: 'get_product' never called. Actual: []
    - tool_use: 'calculate' never called. Actual: []
  [FAIL] percent_off_jacket: Percentage discount composed from primitive operators
    - response_numeric: Expected 71.99 (+/- 0.02), found: ['20', ',']
    - tool_use: 'get_produ

'eval_results/eval_default_20260828_101121.json'

### Inspecting results

The summary tells you *what* passed and failed. To understand *why*, inspect individual tasks.

For any failed task, look at:
- What did the agent actually respond with?
- What tool calls did it make? With what arguments?
- Where did it go wrong: tool selection, argument formatting, or the final answer?

For passing tasks, ask: is it passing for the right reasons? Could it be passing by luck?

For failing tasks, ask: does it feel fair?

In [10]:
# Replace with a task ID you want to inspect
inspect_task(results, "price_jeans")

[PASS] price_jeans: Direct price lookup for jeans
Query: How much do jeans cost?
Response: Jeans cost **$49.99**.

Grades:
  + response_contains: Found '49.99' in response
  + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}

Metrics: {'time': 1.434, 'tool_calls': 1, 'turns': 2, 'input_tokens': 1424, 'output_tokens': 80}

Transcript:
  How much do jeans cost?
  [text] I'll look up the price of jeans for you.
  [tool_use] get_product({'product': 'jeans'})
  [tool_result] 49.99
  [text] Jeans cost **$49.99**.


### Establishing a baseline

LLMs are non-deterministic — the newest Claude models don't expose sampling parameters like temperature at all, and pinned settings never guaranteed identical outputs anyway. Run the eval multiple times to get a reliable baseline before making changes.

In [11]:
baseline = run_eval(run_agent, tasks, num_runs=5)
print_summary(baseline)

  [1/4] percent_off_jacket: FAIL


  [2/4] multi_item_total: FAIL


  [3/4] price_jeans: PASS


  [4/4] price_tshirt: PASS


  [1/4] multi_item_total: FAIL


...

  [1/4] percent_off_jacket: FAIL


  [2/4] price_tshirt: PASS


  [3/4] price_jeans: PASS


  [4/4] multi_item_total: PASS


EVAL RESULTS: 4 tasks, 5 run(s)

--- Run 1 ---
Overall: 2/4 passed (50%)

By category:
  multi_step: 0/2 (0%)
  product_lookup: 2/2 (100%)

Tasks:
  [PASS] price_jeans: Direct price lookup for jeans
    + response_contains: Found '49.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}
  [PASS] price_tshirt: Hyphenated catalog key — does the agent pass the right argument?
    + response_numeric: Found 24.99 (expected 24.99 +/- 0.005)
    + tool_use: Tool 'get_product' called with matching args: {'product': 't-shirt'}
  [FAIL] multi_item_total: Two lookups plus multi-step arithmetic
    - response_numeric: Expected 139.95 (+/- 0.01), found: [',', '1.', '2.', ',']
    - tool_use: 'get_product' never called. Actual: []
    - tool_use: 'calculate' never called. Actual: []
  [FAIL] percent_off_jacket: Percentage discount composed from primitive operators
    - response_numeric: Expected 71.99 (+/- 0.02), found: ['20', ',']
    - tool_use: 'get_p

### 🧭 Decision log 3 — one pass rate is the wrong headline

`print_summary` gives a pass rate per run. Run it five times and you get five numbers, and the natural move is to average them — which throws away the only thing k=5 bought you.

A task that passes 3 times in 5 and a task that passes 0 times in 5 are both "below 100%". One is **flaky** and one is **reliably broken**, they have completely different causes, and you fix them in completely different ways. The average hides which you have.

So the next cell replaces the headline with two metrics, and reads each task with the one its suite calls for:

- **pass@k** — passed at least once. The *capability* question: can the agent do this at all? Rising pass@k means the behaviour is being learned.
- **pass^k** — passed every time. The *reliability* question. This is the number that matters for anything gated: a property that must always hold cannot be certified by "it managed it once".

Gating a regression case on pass@k is a bug, not a shortcut — it lets a must-never-fail property fail one trial in five and still show green.

**What k=5 can and can't tell you.** It can separate flaky from broken. It cannot give you a tight confidence interval — five trials on a binary outcome is a coarse instrument, and a task at 4/5 versus 5/5 is not a distinction worth defending. I'm using it to sort tasks into three buckets (solid / flaky / broken), not to claim precision. Jackdaws' `baselines-and-measurement.md` goes further into interval width if you need the number to survive an argument.

**Run errors are excluded from the denominator.** An API timeout is a failed *run*, not a failed *task* — the distinction cell 12 draws. Counting infrastructure noise as agent error would make every rung of the ablation ladder noisier than the effect it's measuring.

In [12]:
# ── pass@k and pass^k ─────────────────────────────────────────────────────────
# `print_summary` reports one pass rate per run. With k=5 that's five numbers that
# average away the thing we actually want to know: is a failing task *flaky* or
# *reliably broken*? Those look identical in a mean and mean opposite things.
#
#   pass@k  = passed at least once in k trials  -> capability. Can it do this at all?
#   pass^k  = passed on every one of k trials   -> reliability. Can it be relied on?
#
# Jackdaws reads capability cases at pass@k and regression cases at pass^k, and treats
# gating a regression case on pass@k as a bug: a property that must always hold cannot
# be certified by "it managed it once".

SUITE_METRIC = {"capability": "pass@k", "regression": "pass^k"}


def task_stats(results):
    """Per-task aggregation across every run in a results object."""
    by_id, order = {}, []
    for run in results["runs"]:
        for r in run:
            if r["task_id"] not in by_id:
                by_id[r["task_id"]] = []
                order.append(r["task_id"])
            by_id[r["task_id"]].append(r)

    stats = {}
    for tid in order:
        trials = by_id[tid]
        # An API error is a failed *run*, not a failed task (cell 12). Excluded from the
        # denominator and surfaced separately, so infrastructure noise never reads as
        # "the agent got it wrong" — Jackdaws PI012.
        graded = [t for t in trials if not t.get("error")]
        errors = len(trials) - len(graded)
        passes = sum(1 for t in graded if t["passed"])
        k = len(graded)
        task = next((t for t in FULL_SUITE if t["id"] == tid), {})
        stats[tid] = {
            "suite": task.get("suite", "capability"),
            "category": trials[0].get("category", ""),
            "k": k, "passes": passes, "errors": errors,
            "pass_at_k": 1.0 if passes > 0 else 0.0,
            "pass_hat_k": 1.0 if k > 0 and passes == k else 0.0,
            "rate": passes / k if k else 0.0,
        }
    return stats


def passk_table(results, title="", stats=None):
    """Print per-task pass@k / pass^k, and the headline each suite is judged on."""
    stats = stats or task_stats(results)
    k = results["config"].get("num_runs", 1)
    print(f"{'=' * 74}\n{title or 'pass@k / pass^k'}   (k = {k})\n{'=' * 74}")
    print(f"{'task':<24}{'suite':<12}{'passed':>8}{'pass@k':>9}{'pass^k':>9}   verdict")
    print("-" * 74)
    for tid, s in stats.items():
        judged_on = SUITE_METRIC[s["suite"]]
        met = s["pass_at_k"] if judged_on == "pass@k" else s["pass_hat_k"]
        # "flaky" is only visible because we kept the two metrics apart.
        verdict = ("ok" if met else "FAILS") + (
            "  (flaky)" if s["pass_at_k"] and not s["pass_hat_k"] else "")
        err = f"  [{s['errors']} run error(s)]" if s["errors"] else ""
        print(f"{tid:<24}{s['suite']:<12}{s['passes']:>4}/{s['k']:<3}"
              f"{s['pass_at_k']:>9.2f}{s['pass_hat_k']:>9.2f}   {verdict}{err}")
    print("-" * 74)

    for suite, metric in SUITE_METRIC.items():
        rows = [s for s in stats.values() if s["suite"] == suite]
        if not rows:
            continue
        key = "pass_at_k" if metric == "pass@k" else "pass_hat_k"
        met = sum(r[key] for r in rows)
        print(f"{suite:<12} judged on {metric:<7} {met:.0f}/{len(rows)} tasks meet it")
    print()
    return stats


baseline_stats = passk_table(baseline, "BASELINE A — agent as shipped, mechanical tasks only")

BASELINE A — agent as shipped, mechanical tasks only   (k = 5)
task                    suite         passed   pass@k   pass^k   verdict
--------------------------------------------------------------------------
price_jeans             regression     5/5       1.00     1.00   ok
price_tshirt            capability     5/5       1.00     1.00   ok
multi_item_total        capability     2/5       1.00     0.00   ok  (flaky)
percent_off_jacket      capability     0/5       0.00     0.00   FAILS
--------------------------------------------------------------------------
capability   judged on pass@k  2/3 tasks meet it
regression   judged on pass^k  1/1 tasks meet it



---

## Part 5: Improve the Agent

Use what you learned from the eval results to improve the agent.

Things you can change:
- **Tool descriptions.** The current specs are intentionally vague. What information would help Claude use the tools correctly?
- **System prompt.** `"You are a helpful assistant."` gives Claude zero context about being a shopping assistant.
- **Error handling.** What happens when a product isn't found? When an invalid operator is used?
- **Tool implementation.** Could the tool be more forgiving about input format?

After making changes, **scroll back up and re-run the agent cell**, then re-run the eval to see if your changes helped.

> **Note on where this got done.** I did *not* edit the agent cell in place — that would have destroyed the Part 4 baseline and made the before/after unreproducible. The measured version of this section is **Part 5b**, below Part 6: four agent configurations built as separate closures and scored on the full frozen corpus, one variable per rung. It sits after Part 6 because three of the seven tasks need the judge that Part 6 builds.


---

## Part 6: Add an LLM-as-Judge Grader

Some queries can't be graded with a deterministic grader. Consider:
- *"What do you sell?"* There are many valid ways to describe the catalog.
- *"Which is a better deal, 2 shirts or 1 jacket?"* The answer requires reasoning, not just a number.
- *"I have $100, what should I buy?"* Any sensible recommendation is a valid answer.

For these, we need an **LLM-as-judge**: a grader that uses Claude itself to evaluate the response.

### The contract

The grader interface is the same as the deterministic graders:

```python
def grade_llm_judge(result, check, context=None):
    # check: a string describing what to evaluate, e.g.
    #   "Response lists available product categories"
    #   "Response correctly identifies the better deal with reasoning"
    #
    # context: includes the original query (context["query"]) and task metadata
    #
    # Returns: {"score": 0.0 or 1.0, "reason": "..."}
```

Each check is a natural-language criterion. The grader sends the agent's response + the criterion to Claude and asks for a binary pass/fail judgment.

### Implementation hints

- **Prompt structure.** Give the judge the original query, the agent's response (`result["final_text"]`), and the criterion (`check`). Ask it to evaluate whether the criterion is met.
- **Force a structured answer.** Ask the judge to respond with `PASS` or `FAIL` on the first line, followed by a reason. This makes parsing straightforward.
- **Keep it atomic.** Each check is a single criterion. Don't ask one LLM call to evaluate multiple criteria at once.

### Implement it

Fill in the grader below, register it, then write tasks that use it.

In [13]:
# Implement the LLM-as-judge grader
#
# We use structured outputs (output_config.format) so the verdict comes back as
# validated JSON with an enum — no string-parsing a free-text "PASS"/"FAIL", and
# no sampling params (temperature is removed on the newest models; determinism
# comes from the schema, not the dial).

JUDGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "verdict": {"type": "string", "enum": ["PASS", "FAIL"]},
            "reason": {"type": "string", "description": "One sentence."},
        },
        "required": ["verdict", "reason"],
        "additionalProperties": False,
    },
}

# ── Judge model: deliberately NOT the model under test ────────────────────────
# Cell 6 set MODEL = FAST_MODEL, so the agent being graded runs on Haiku. A Haiku
# judge would be grading its own family, and a model grading its own output rates it
# higher than an independent judge would — self-preference bias, quantified at 10–25%
# for same-model judge panels. Pinning the judge to a different tier is the cheap
# mitigation; a heterogeneous panel is the expensive one.
try:
    JUDGE_MODEL = _model("claude-sonnet-5")   # `_model` resolves Anthropic vs Bedrock ids
except NameError:                             # cells run out of order — fall back
    JUDGE_MODEL = "claude-sonnet-5"

JUDGE_PROMPT = """You are grading one response from a shopping assistant against a single criterion.

<user_query>
{query}
</user_query>

<assistant_response>
{response}
</assistant_response>

<criterion>
{criterion}
</criterion>

Judge ONLY this criterion. Do not reward or punish anything else about the response.
If the response does not contain enough to satisfy the criterion, the verdict is FAIL.
Answer with the verdict and one sentence of reasoning."""


def grade_llm_judge(result, check, context=None):
    """Judge one atomic criterion. Returns {"score": 0.0|1.0, "reason": str}.

    The judge sees the query, the response and the criterion — and nothing else. It is
    never told the catalog or the expected answer, because a grader that knows something
    the agent was denied stops measuring the agent and starts measuring the leak
    (Jackdaws PI009).
    """
    prompt = JUDGE_PROMPT.format(
        query=(context or {}).get("query", ""),
        response=result.get("final_text", "") or "(the assistant returned no text)",
        criterion=check,
    )
    kwargs = {"model": JUDGE_MODEL, "max_tokens": 256,
              "messages": [{"role": "user", "content": prompt}]}

    # Every parse path is guarded (PI010). A judge that raises would abort the trial;
    # a judge that silently returns a default would quietly award or deny a point.
    # Both failure modes end up here as an explicit 0.0 with the cause in `reason`.
    try:
        try:
            response = client.messages.create(**kwargs, output_config={"format": JUDGE_SCHEMA})
            raw = response.content[0].text
            data = json.loads(raw)
            verdict, reason = data["verdict"], data["reason"]
        except (TypeError, anthropic.BadRequestError) as schema_exc:
            # Older SDK or a model without structured outputs: fall back to a parsed
            # first line. Recorded in the reason so a fallback never passes unnoticed.
            response = client.messages.create(
                **{**kwargs, "messages": [{"role": "user", "content":
                    prompt + "\n\nReply with PASS or FAIL on the first line, then one sentence."}]}
            )
            text = "".join(b.text for b in response.content if isinstance(b, TextBlock)).strip()
            head, _, tail = text.partition("\n")
            verdict = "PASS" if head.strip().upper().startswith("PASS") else "FAIL"
            reason = f"[text fallback: {type(schema_exc).__name__}] {tail.strip() or head.strip()}"
    except Exception as exc:
        return {"score": 0.0, "reason": f"judge error: {type(exc).__name__}: {exc}"}

    return {"score": 1.0 if verdict == "PASS" else 0.0, "reason": reason}


# Register it so the runner can use it
GRADER_REGISTRY["llm_judge"] = grade_llm_judge

print(f"llm_judge registered.  agent under test: {MODEL}   judge: {JUDGE_MODEL}")

llm_judge registered.  agent under test: claude-haiku-4-5   judge: claude-sonnet-5


### 🧭 Decision log 4 — designing the judge, and proving it can fail

**Structured output, not string parsing.** The notebook scaffolds `output_config.format` and it's the right call: the verdict comes back as schema-validated JSON with an enum, so there's no regex over a free-text `PASS`/`FAIL` and no "the model said `Pass:` with a colon" edge case. The `try/except` fallback to text parsing is there for an older SDK or a model without structured outputs — and it *labels itself* in the reason string, so a silent degradation to the weaker path can't pass unnoticed.

**Every parse path is guarded.** A judge that raises aborts the trial; a judge that silently returns a default quietly awards or denies a point. Both end up as an explicit `0.0` with the cause in `reason` (Jackdaws PI010). You can see which is which when you read the results.

**The judge runs on Sonnet; the agent runs on Haiku.** This is the D5 finding from decision log 1 paying off. Cell 6 quietly sets `MODEL = FAST_MODEL`, so the agent under test is Haiku — and the obvious judge model, `FAST_MODEL`, would have been the same family grading its own output. That's self-preference bias, measured at 10–25% for same-model judge panels. Pinning the judge a tier up is the cheap mitigation. (The expensive one is a heterogeneous panel with a majority vote; overkill at this scale.)

**The judge is catalog-blind.** It sees the query, the response, and one criterion. It is never shown the price list or the expected answer. Jackdaws PI009: a grader that knows something the agent was denied stops measuring the agent and starts measuring the leak. Concretely — if I handed the judge the catalog, `catalog_overview` would silently become "did the agent guess my list", not "did the agent describe the shop".

**One criterion per call.** `unknown_product_shoes` has two criteria and makes two calls. Bundling them into one judgment saves an API call and costs you the ability to know *which half* failed.

**No caching.** Judging the same response text twice costs two calls, and I considered memoising on `(criterion, response)`. I didn't, because trials within a run are supposed to be independent — a cache would silently remove judge variance from repeated trials and make pass^k look better than it is.

**Then: the control cell.** No judged number gets quoted until every grader has been *watched* producing both verdicts, on inputs built to earn them. This is the single highest-value cell in the notebook and it costs two API calls.

In [14]:
# ── Negative controls: prove every grader can fail ────────────────────────────
# "A grader that has never been observed to fail is not a measurement. It is a constant
# with a plausible-looking name." (Jackdaws PI007)
#
# The failure this catches is not hypothetical. The module that owns this rule documents
# three real "validators" that scored a perfect 10 on empty strings, on refusals, and on
# "TODO" — every one of them looked reasonable in isolation, and every one of them turned
# out to accept nearly anything. In a results table that is indistinguishable from a
# well-behaved agent.
#
# So: each grader gets a *pair* — an input built to earn a FAIL and one built to earn a
# PASS. A grader that can only ever return one of the two verdicts is not measuring.

def _fake(final_text="", tool_calls=()):
    """Minimal `result` shaped the way the runner hands it to a grader."""
    return {"final_text": final_text,
            "tool_calls": [dict(c) for c in tool_calls],
            "messages": []}


_GP_JEANS = {"name": "get_product", "arguments": {"product": "jeans"}, "id": "t1"}
_GP_TSHIRT_WRONG = {"name": "get_product", "arguments": {"product": "tshirt"}, "id": "t2"}
_CALC = {"name": "calculate", "arguments": {"op": "*", "input1": 2, "input2": 3}, "id": "t3"}

# (grader, check, result, expected_score, what this control proves)
CONTROLS = [
    ("response_contains", "49.99", _fake("Jeans are on sale this week."), 0.0,
     "missing string must FAIL"),
    ("response_contains", "49.99", _fake("Jeans are 49.99."), 1.0,
     "present string must PASS"),

    ("response_numeric", {"value": 49.99, "tolerance": 0.01}, _fake("That comes to 12.34."), 0.0,
     "wrong number must FAIL"),
    ("response_numeric", {"value": 49.99, "tolerance": 0.01}, _fake("That comes to $49.99."), 1.0,
     "right number must PASS"),
    ("response_numeric", {"value": 49.99, "tolerance": 0.01}, _fake("I could not find that item."), 0.0,
     "no number at all must FAIL (not crash)"),

    ("tool_use", {"tool_name": "get_product"}, _fake("Jeans are 49.99.", []), 0.0,
     "answer with no tool call must FAIL (ungrounded)"),
    ("tool_use", {"tool_name": "get_product"}, _fake("Jeans are 49.99.", [_GP_JEANS]), 1.0,
     "matching tool call must PASS"),
    ("tool_use", {"tool_name": "get_product", "arguments": {"product": "t-shirt"}},
     _fake("...", [_GP_TSHIRT_WRONG]), 0.0,
     "wrong argument must FAIL — this is the check price_tshirt leans on"),

    ("tool_call_count", {"max": 0}, _fake("...", [_GP_JEANS, _CALC]), 0.0,
     "tools used on an out-of-scope query must FAIL"),
    ("tool_call_count", {"max": 0}, _fake("That's a question for a philosopher.", []), 1.0,
     "no tools used must PASS"),

    ("llm_judge",
     "The response does not claim to sell product categories outside clothing and accessories "
     "(for example food, electronics or furniture).",
     _fake("We sell fresh produce, engine parts and garden furniture."), 0.0,
     "judge must FAIL an answer that plainly violates the criterion"),
    ("llm_judge",
     "The response names at least three specific clothing or accessory items that the shop sells.",
     _fake("We stock jeans, shirts, jackets, hats and socks."), 1.0,
     "judge must PASS an answer that plainly satisfies it"),
]

print(f"{'grader':<20}{'expect':<9}{'got':<9}{'':<4}what it proves")
print("-" * 96)
_broken = []
for _grader, _check, _result, _expected, _why in CONTROLS:
    _ctx = {"query": "control", "task_id": "control", "model": None}
    _grade = GRADER_REGISTRY[_grader](_result, _check, _ctx)
    _ok = _grade["score"] == _expected
    if not _ok:
        _broken.append((_grader, _why, _grade["reason"]))
    print(f"{_grader:<20}{_expected:<9.1f}{_grade['score']:<9.1f}{'ok ' if _ok else 'XX ':<4}{_why}")
print("-" * 96)

if _broken:
    for _g, _why, _reason in _broken:
        print(f"  ! {_g}: {_why}\n    grader said: {_reason}")
    raise AssertionError(
        f"{len(_broken)} grader control(s) failed — a grader that cannot produce both "
        "verdicts is not a measurement. Fix it before trusting any number below."
    )
print(f"\nAll {len(CONTROLS)} controls behaved. Every grader — the judge included — has now "
      "been observed producing both verdicts.")

grader              expect   got          what it proves
------------------------------------------------------------------------------------------------
response_contains   0.0      0.0      ok  missing string must FAIL
response_contains   1.0      1.0      ok  present string must PASS
response_numeric    0.0      0.0      ok  wrong number must FAIL
response_numeric    1.0      1.0      ok  right number must PASS
response_numeric    0.0      0.0      ok  no number at all must FAIL (not crash)
tool_use            0.0      0.0      ok  answer with no tool call must FAIL (ungrounded)
tool_use            1.0      1.0      ok  matching tool call must PASS
tool_use            0.0      0.0      ok  wrong argument must FAIL — this is the check price_tshirt leans on
tool_call_count     0.0      0.0      ok  tools used on an out-of-scope query must FAIL
tool_call_count     1.0      1.0      ok  no tools used must PASS


llm_judge           0.0      0.0      ok  judge must FAIL an answer that plainly violates the criterion


llm_judge           1.0      1.0      ok  judge must PASS an answer that plainly satisfies it
------------------------------------------------------------------------------------------------

All 12 controls behaved. Every grader — the judge included — has now been observed producing both verdicts.


In [15]:
# Add tasks that use the LLM-as-judge grader
#
# They were written and frozen back in Part 3, along with everything else in the corpus —
# only the grader that scores them arrived here. Adding cases *after* seeing a baseline is
# how a corpus drifts, so this cell wires up what already exists rather than inventing more.
llm_judge_tasks = JUDGED_TASKS

all_tasks = tasks + llm_judge_tasks
assert corpus_fingerprint(all_tasks) == CORPUS_SHA, "corpus drifted since it was frozen"

# One trial each — a smoke test that the judged tasks execute and that the verdicts read
# sensibly. The measured baseline is Baseline B in Part 5b, at k=5 on this same suite.
judged_smoke = run_eval(run_agent, all_tasks, num_runs=1, max_workers=MAX_WORKERS)
print_summary(judged_smoke)

  [1/7] percent_off_jacket: FAIL


  [2/7] price_jeans: PASS


  [3/7] price_tshirt: PASS


  [4/7] multi_item_total: PASS


  [5/7] unknown_product_shoes: PASS


  [6/7] catalog_overview: FAIL


  [7/7] offtopic_redirect: PASS


EVAL RESULTS: 7 tasks, 1 run(s)

Overall: 5/7 passed (71%)

By category:
  capabilities: 0/1 (0%)
  graceful_degradation: 1/1 (100%)
  multi_step: 1/2 (50%)
  out_of_scope: 1/1 (100%)
  product_lookup: 2/2 (100%)

Tasks:
  [PASS] price_jeans: Direct price lookup for jeans
    + response_contains: Found '49.99' in response
    + tool_use: Tool 'get_product' called with matching args: {'product': 'jeans'}
  [PASS] price_tshirt: Hyphenated catalog key — does the agent pass the right argument?
    + response_numeric: Found 24.99 (expected 24.99 +/- 0.005)
    + tool_use: Tool 'get_product' called with matching args: {'product': 't-shirt'}
  [PASS] multi_item_total: Two lookups plus multi-step arithmetic
    + response_numeric: Found 139.95 (expected 139.95 +/- 0.01)
    + tool_use: Tool 'get_product' was called
    + tool_use: Tool 'calculate' was called
  [FAIL] percent_off_jacket: Percentage discount composed from primitive operators
    - response_numeric: Expected 71.99 (+/- 0.02), fou

---

## Part 5b: Improving the Agent — the ablation ladder

> **Why this section sits after Part 6.** Part 5 tells you to scroll up, edit the agent cell and re-run. I've done the measured version down here instead, for two reasons. First, the ladder has to be scored against the **full frozen corpus**, and three of those seven tasks need the judge that Part 6 builds. Second, editing the agent cell in place destroys the baseline: once you've overwritten `SYSTEM_PROMPT`, the committed notebook can no longer show the before state next to the after. Everything below leaves cell 6 untouched.

Four measured runs, each changing exactly one thing:

| run | `changed` | what moves |
|---|---|---|
| **Baseline B** | — | the agent exactly as shipped |
| **V1** | `system_prompt` | D4 — give it a role, a scope, and an instruction to ground prices in tool calls |
| **V2** | `tool_specs` | D2 — real descriptions, and an `enum` of what's actually stocked |
| **V3** | `tool_implementations` | D1 + D3 — normalise the lookup, and make every failure a result that says what to do next |

Each rung inherits the one above it, so V3 is all four fixes and the deltas between rungs are the attributable part.

In [16]:
# ── Harness hardening ─────────────────────────────────────────────────────────
# Everything measured in this section runs through one instrument, so the rungs stay
# comparable. Three additions over cell 6's agent, none of which change its behaviour:
#
#   1. Config as parameters, not module globals — so v0..v3 coexist in one kernel and
#      the baseline stays re-runnable. Cell 6 is never edited.
#   2. A turn cap (D6). An agent that loops is an agent defect, so hitting the cap is a
#      task FAILURE — graders see whatever text exists and score it.
#   3. Truncation detection (D6). `max_tokens` running out is our budget being too small,
#      not the agent being wrong, so it's raised and lands in the runner's `error` field —
#      a failed RUN, excluded from the denominator (cell 12's distinction; Jackdaws PI012).

MAX_TURNS = 8


class TruncatedRun(RuntimeError):
    """max_tokens ran out mid-answer: a harness-budget failure, not a wrong answer."""


def default_execute(tool_registry, name, inputs):
    """Cell 6's behaviour, verbatim, so v0 really is the shipped agent."""
    try:
        return str(tool_registry[name](**inputs))
    except Exception as e:
        return f"Error: {e}"


def build_agent(system_prompt, tool_specs, tool_registry, label, execute=default_execute):
    """Return an agent_fn with the signature the eval runner expects."""

    def agent_fn(prompt, eval_mode=False, model=None):
        messages = [{"role": "user", "content": prompt}]
        total_in = total_out = turns = 0

        while True:
            response = client.messages.create(
                model=model or MODEL,
                system=system_prompt,
                max_tokens=1024,
                tools=tool_specs,
                messages=messages,
            )
            total_in += response.usage.input_tokens
            total_out += response.usage.output_tokens
            messages.append({"role": "assistant", "content": response.content})
            turns += 1

            if response.stop_reason == "max_tokens":
                raise TruncatedRun(
                    f"{label}: hit max_tokens on turn {turns} — the answer is cut off, so "
                    "grading it would score our token budget, not the agent."
                )
            if response.stop_reason != "tool_use":
                break
            if turns >= MAX_TURNS:
                break  # agent defect — fall through and let the graders fail it

            tool_results = [
                {"type": "tool_result", "tool_use_id": b.id,
                 "content": execute(tool_registry, b.name, b.input)}
                for b in response.content if isinstance(b, ToolUseBlock)
            ]
            messages.append({"role": "user", "content": tool_results})

        if eval_mode:
            return {"messages": messages,
                    "usage": {"input_tokens": total_in, "output_tokens": total_out}}
        return "\n".join(b.text for b in response.content if isinstance(b, TextBlock))

    agent_fn.label = label
    return agent_fn


# ── Run fingerprints (PI002 / PI004) ──────────────────────────────────────────
def run_measured(agent_fn, suite, changed, baseline_run=None, k=None, model=None):
    """run_eval, plus a record of what this run actually was.

    `changed` is the one variable that moved since `baseline_run`. A run with no
    `changed` field is a run nobody can later audit for a confound, so it's required.
    """
    k = k or K
    results = run_eval(agent_fn, suite, model=model, num_runs=k, max_workers=MAX_WORKERS)
    results["config"].update({
        "corpus_sha": corpus_fingerprint(suite),
        "agent_version": agent_fn.label,
        "agent_model": model or MODEL,
        "judge_model": JUDGE_MODEL,
        "k": k,
        "max_workers": MAX_WORKERS,
        "changed": changed,
        "baseline_run": baseline_run,
    })
    return results


COMPARABLE_KEYS = ("corpus_sha", "agent_model", "k", "max_workers", "judge_model")


def compare_runs(before, after):
    """Refuse to diff two runs that differ in more than the declared variable (PI003)."""
    diffs = [key for key in COMPARABLE_KEYS
             if before["config"].get(key) != after["config"].get(key)]
    declared = after["config"].get("changed")
    if diffs:
        raise ValueError(
            f"runs are not comparable — {diffs} differ, but the declared change was "
            f"'{declared}'. Either the change was bigger than claimed, or the harness "
            "moved underneath the measurement."
        )
    if not declared:
        raise ValueError("the 'after' run declares no changed variable — nothing to attribute a delta to")
    return True


print("Hardened harness ready: build_agent / run_measured / compare_runs")

Hardened harness ready: build_agent / run_measured / compare_runs


In [17]:
# ── The four configurations ───────────────────────────────────────────────────
import unicodedata

# The catalog, lifted out of `get_product`'s body so the tool SPEC and the tool
# IMPLEMENTATION can share one source of truth. Same 12 items, same prices.
CATALOG = {
    "jeans": 49.99, "shirt": 29.99, "dress": 59.99, "jacket": 89.99,
    "sneakers": 74.99, "hat": 19.99, "socks": 9.99, "hoodie": 44.99,
    "shorts": 34.99, "t-shirt": 24.99, "sweater": 54.99, "belt": 24.99,
}

# ── V1 — the system prompt (D4) ───────────────────────────────────────────────
# "You are a helpful assistant" tells Claude nothing about the job. This gives it a
# role, a scope boundary, and the two rules that matter for a tool-using agent:
# ground every fact in a tool call, and delegate every calculation.
SYSTEM_PROMPT_V1 = """You are the shopping assistant for a small clothing boutique.

You help customers with two things: what items cost, and arithmetic on those prices
(totals, discounts, comparisons).

Rules:
- Never state a price from memory. Every price you give must come from a `get_product`
  call in this conversation.
- Never do arithmetic in your head. Route every calculation through `calculate`, even
  simple ones — including multi-step ones, one operation at a time.
- If an item is not in the catalog, say so plainly and offer the closest thing we do
  stock. Do not invent a price for something we do not sell.
- For anything outside shopping, answer briefly and do not call any tools.

Keep answers short and lead with the number the customer asked for."""

# ── V2 — the tool specs (D2) ──────────────────────────────────────────────────
# The shipped specs describe each tool with its own name. A tool spec is the contract
# the model sees, and this one said nothing: not what's stocked, not what argument
# format is valid, not what happens on a miss. Rewritten per the Jackdaws
# `tools-and-permissions.md` guidance — a description says when to use the tool, what
# arguments are valid, and how it fails; constrained arguments get an `enum`.
GET_PRODUCT_SPEC_V2 = {
    "name": "get_product",
    "description": (
        "Look up the current price of one item in the boutique catalog. "
        "Call this once per distinct item before quoting any price — prices are never "
        "to be recalled from memory. "
        "If the customer names something not in the enum (for example 'shoes'), the "
        "closest stocked item is usually the right suggestion, but do not quote its "
        "price as though it were the item they asked for."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "product": {
                "type": "string",
                "description": "The catalog item to price. Must be one of the listed values.",
                # An enum is how a constrained argument stops being a guessing game.
                # It also, on its own, tells the model what the shop sells.
                "enum": sorted(CATALOG),
            },
        },
        "required": ["product"],
        "additionalProperties": False,
    },
}

CALCULATE_SPEC_V2 = {
    "name": "calculate",
    "description": (
        "Apply one arithmetic operation to two numbers and return the result. "
        "Use this for every calculation rather than computing in your head. "
        "There is no percentage operator: express a 20% discount as a multiplication "
        "by 0.8, and build multi-step totals with one call per step."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "op": {
                "type": "string",
                "description": "The operation to apply.",
                "enum": ["+", "-", "*", "/", "**"],
            },
            "input1": {"type": "number", "description": "Left-hand operand."},
            "input2": {"type": "number", "description": "Right-hand operand."},
        },
        "required": ["op", "input1", "input2"],
        "additionalProperties": False,
    },
}

TOOL_SPECS_V2 = [GET_PRODUCT_SPEC_V2, CALCULATE_SPEC_V2]

# ── V3 — the tool implementations (D1 + D3) ───────────────────────────────────
# Jackdaws `tools-and-permissions.md`, § Error handling: "Every failure is a result…
# The error should include safe next steps." A bare KeyError surfacing as `Error: 'shoes'`
# is neither. Two changes, one coherent variable:
#   D1  `get_product` normalises the argument, and a miss returns a `not_found` result
#       that names what IS stocked and what to do about it.
#   D3  `calculate` rejects an unknown operator instead of returning None -> "None".

def _normalise(name):
    """Casefold, collapse whitespace, and unify the four unicode dashes to ASCII '-'."""
    s = unicodedata.normalize("NFKD", str(name)).strip().lower()
    for dash in ("‐", "‑", "–", "—"):
        s = s.replace(dash, "-")
    return re.sub(r"\s+", " ", s)


def _build_index(catalog):
    """Map every plausible spelling of a key onto that key: hyphen/space/joined, singular
    and plural. 'T-Shirts', 't shirt' and 'tshirt' all have to reach 't-shirt'."""
    index = {}
    for key in catalog:
        forms = {key, key.replace("-", " "), key.replace("-", "")}
        forms |= {f + "s" for f in set(forms)}
        for form in forms:
            index[_normalise(form)] = key
    return index


CATALOG_INDEX = _build_index(CATALOG)


def get_product_v3(product):
    key = CATALOG_INDEX.get(_normalise(product))
    if key is None:
        return {
            "error": "not_found",
            "requested": product,
            "message": f"'{product}' is not in the catalog.",
            "available": sorted(CATALOG),
            "next_steps": ("Tell the customer we do not stock this, and offer the closest "
                           "item from `available`. Do not quote a price for an item that "
                           "is not listed."),
        }
    return {"product": key, "price": CATALOG[key], "currency": "USD"}


_OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b,
        "*": lambda a, b: a * b, "/": lambda a, b: a / b,
        "**": lambda a, b: a ** b}


def calculate_v3(op, input1, input2):
    if op not in _OPS:
        return {"error": "invalid_arguments",
                "message": f"unknown operator {op!r}",
                "allowed_ops": sorted(_OPS),
                "next_steps": "Re-issue with one of allowed_ops; express percentages as multiplication."}
    if op == "/" and input2 == 0:
        return {"error": "invalid_arguments", "message": "division by zero",
                "next_steps": "Check the divisor before retrying."}
    return {"op": op, "input1": input1, "input2": input2, "result": _OPS[op](input1, input2)}


TOOL_REGISTRY_V3 = {"get_product": get_product_v3, "calculate": calculate_v3}


def execute_v3(tool_registry, name, inputs):
    """Every failure is a result, in the same JSON shape as a success."""
    fn = tool_registry.get(name)
    if fn is None:
        return json.dumps({"error": "unknown_tool", "tool": name,
                           "available": sorted(tool_registry)})
    try:
        return json.dumps(fn(**inputs))
    except TypeError as exc:      # wrong/missing arguments against the signature
        return json.dumps({"error": "invalid_arguments", "message": str(exc)})
    except Exception as exc:
        return json.dumps({"error": "internal_error", "message": f"{type(exc).__name__}: {exc}"})


# ── The ladder: each rung adds exactly one of the above ───────────────────────
agent_v0 = build_agent(SYSTEM_PROMPT,    ALL_TOOL_SPECS, TOOL_REGISTRY,    "v0")
agent_v1 = build_agent(SYSTEM_PROMPT_V1, ALL_TOOL_SPECS, TOOL_REGISTRY,    "v1")
agent_v2 = build_agent(SYSTEM_PROMPT_V1, TOOL_SPECS_V2,  TOOL_REGISTRY,    "v2")
agent_v3 = build_agent(SYSTEM_PROMPT_V1, TOOL_SPECS_V2,  TOOL_REGISTRY_V3, "v3", execute=execute_v3)

# v0 must be the shipped agent, or Baseline B measures something other than the baseline.
assert (SYSTEM_PROMPT, ALL_TOOL_SPECS, TOOL_REGISTRY) == \
       (SYSTEM_PROMPT, ALL_TOOL_SPECS, TOOL_REGISTRY), "config identity"
assert agent_v0.label == "v0" and ALL_TOOL_SPECS is not TOOL_SPECS_V2
print("v0 = shipped config (system prompt, specs and registry are cell 6's own objects)")
print(f"v1 changes: system_prompt        ({len(SYSTEM_PROMPT)} -> {len(SYSTEM_PROMPT_V1)} chars)")
print(f"v2 changes: tool_specs           (product enum: {len(CATALOG)} items)")
print(f"v3 changes: tool_implementations (normalised lookup + structured errors)")
print(f"\nSanity: get_product_v3('T-Shirts') -> {get_product_v3('T-Shirts')}")
print(f"        get_product_v3('shoes')    -> {get_product_v3('shoes')['error']}, "
      f"{len(get_product_v3('shoes')['available'])} alternatives offered")
print(f"        calculate_v3('%', 1, 2)    -> {calculate_v3('%', 1, 2)['error']}")

v0 = shipped config (system prompt, specs and registry are cell 6's own objects)
v1 changes: system_prompt        (28 -> 748 chars)
v2 changes: tool_specs           (product enum: 12 items)
v3 changes: tool_implementations (normalised lookup + structured errors)

Sanity: get_product_v3('T-Shirts') -> {'product': 't-shirt', 'price': 24.99, 'currency': 'USD'}
        get_product_v3('shoes')    -> not_found, 12 alternatives offered
        calculate_v3('%', 1, 2)    -> invalid_arguments


### 🧭 Decision log 5 — one variable per run

*Jackdaws `prompt-evaluation-best-practices`: PI002 run fingerprints · PI003 compare only comparable runs · PI004 one variable per run · PI012 distinguish failed runs from failed tasks*

**Why a ladder and not a before/after.** The obvious way to do Part 5 is to fix all four defects at once, re-run, and report "62% → 91%". That number is true and useless. It tells you the bundle helped; it cannot tell you *which part* helped, whether any part hurt, or what to keep if you later need to cut one. PI004 is the rule: **one variable per run.** Four rungs cost four times the API calls and buy four attributable deltas instead of one unattributable one.

**What counts as "one variable" here.** V3 changes two functions at once — `get_product` gains normalisation *and* a structured `not_found` result; `calculate` rejects unknown operators. That is deliberate, and it is still one variable: the variable is **the tool result format**. Normalising the argument without also fixing the error return would produce a config nobody would ship (a miss would still surface as `Error: 'shoes'`), and measuring configs nobody would ship is how a ladder turns into busywork. The line to hold is that a rung must be a coherent change you'd actually make, not that it must touch exactly one line.

**What's in a fingerprint.** `run_measured` stamps every result set with `corpus_sha`, `agent_version`, `agent_model`, `judge_model`, `k`, `max_workers`, `changed` and `baseline_run`. `compare_runs` then *refuses* to compute a delta if any of the first five differ — PI003, enforced in code rather than in a comment. This is the guard that catches the mistake you actually make at 6pm: adding a task, re-running one arm, and diffing it against yesterday's other arm. The fingerprint makes that combination raise instead of print.

**Why cell 6 is untouched.** The exercise's own Part 4 output — the run in cell 21 — has to stay reproducible for anyone reading this notebook top to bottom. So the improvements live in `build_agent` closures down here, and `run_agent` above is exactly as shipped. `agent_v0` is built from cell 6's own `SYSTEM_PROMPT`, `ALL_TOOL_SPECS` and `TOOL_REGISTRY` objects, so Baseline B is the shipped agent, measured on the new instrument.

**Baselines A and B are not comparable, and I'm not going to compare them.** Baseline A (cell 21) ran 4 mechanical tasks through the original `run_agent`. Baseline B runs 7 tasks — three of them judged — through `build_agent` with a turn cap. Different corpus, different harness, different grader mix. Their pass rates are two numbers about two different things, and `compare_runs` would reject the pair on `corpus_sha` alone. Baseline A's job was to show the harness works and to find the failure modes. Baseline B is the first rung of the actual measurement.

**Truncation is a failed run, not a wrong answer** (PI012). `build_agent` raises `TruncatedRun` on `stop_reason == "max_tokens"`. `run_single_task` already catches exceptions into the `error` field, and `task_stats` already excludes `error` trials from the pass@k denominator — so a truncated trial is reported as an error and does not quietly depress a score. A turn-cap hit is different: that *is* a task failure (the agent looped), so it breaks out of the loop and returns whatever it has.

In [18]:
# ── Run the ladder ────────────────────────────────────────────────────────────
# 4 configs x 7 tasks x k=5. Each rung declares exactly what moved relative to the
# rung before it; `run_measured` stamps that into results["config"] so `compare_runs`
# can refuse an apples-to-oranges diff later.
assert corpus_fingerprint(FULL_SUITE) == CORPUS_SHA, "corpus changed since it was frozen"

LADDER = [
    ("Baseline B", agent_v0, None,                     None),
    ("V1",         agent_v1, "system_prompt",          "Baseline B"),
    ("V2",         agent_v2, "tool_specs",             "V1"),
    ("V3",         agent_v3, "tool_implementations",   "V2"),
]

ladder_runs = {}
for name, agent_fn, changed, baseline in LADDER:
    print(f"\n{'=' * 70}\n{name}  (changed: {changed or 'nothing — this is the reference'})\n{'=' * 70}")
    ladder_runs[name] = run_measured(agent_fn, FULL_SUITE, changed=changed, baseline_run=baseline)

print("\nAll four rungs complete.")


Baseline B  (changed: nothing — this is the reference)


  [1/7] percent_off_jacket: FAIL


  [2/7] multi_item_total: FAIL


  [3/7] price_tshirt: PASS


  [4/7] price_jeans: PASS


...

  [4/7] multi_item_total: PASS


  [5/7] offtopic_redirect: PASS


  [6/7] unknown_product_shoes: FAIL


  [7/7] catalog_overview: PASS



All four rungs complete.


In [19]:
# ── Read the ladder ───────────────────────────────────────────────────────────
# pass^k per task per rung: "did this task pass on ALL k trials". For a shipping
# decision that is the number that matters — pass@k tells you the capability exists
# somewhere in the distribution, pass^k tells you a customer will actually get it.
ladder_stats = {name: task_stats(res) for name, res in ladder_runs.items()}
NAMES = [row[0] for row in LADDER]
CHANGED = {row[0]: row[2] for row in LADDER}

# Guarded comparisons first — a rung that isn't comparable to its baseline must not
# be quoted as a delta (PI003).
print("Comparability check (PI003):")
for name, _, changed, baseline in LADDER:
    if baseline is None:
        print(f"  {name:<11} reference run")
        continue
    try:
        compare_runs(ladder_runs[baseline], ladder_runs[name])
        print(f"  {name:<11} comparable to {baseline} (only `{changed}` differs) ✓")
    except ValueError as exc:
        print(f"  {name:<11} NOT COMPARABLE to {baseline}: {exc}")

task_order = [t["id"] for t in FULL_SUITE]
suite_of = {t["id"]: t.get("suite", "capability") for t in FULL_SUITE}

print(f"\n{'=' * 86}")
print(f"ABLATION LADDER — pass^{K} per task (k={K} trials each, corpus {CORPUS_SHA})")
print("=" * 86)
header = f"{'task':<24}{'suite':<12}" + "".join(f"{n:>12}" for n in NAMES)
print(header)
print("-" * len(header))
for tid in task_order:
    row = f"{tid:<24}{suite_of[tid]:<12}"
    for name in NAMES:
        st = ladder_stats[name].get(tid)
        row += f"{'—':>12}" if st is None else f"{st['pass_hat_k']:>11.0%} "
    print(row)
print("-" * len(header))

def _suite_mean(stats, metric, suite=None):
    vals = [s[metric] for tid, s in stats.items() if suite is None or suite_of[tid] == suite]
    return sum(vals) / len(vals) if vals else float("nan")

for label, suite in (("all tasks", None), ("capability", "capability"), ("regression", "regression")):
    row = f"{'mean pass^k · ' + label:<36}"
    row += "".join(f"{_suite_mean(ladder_stats[n], 'pass_hat_k', suite):>11.0%} " for n in NAMES)
    print(row)
row = f"{'mean pass@k · all tasks':<36}"
row += "".join(f"{_suite_mean(ladder_stats[n], 'pass_at_k'):>11.0%} " for n in NAMES)
print(row)
row = f"{'run errors (excluded from denom.)':<36}"
row += "".join(f"{sum(s['errors'] for s in ladder_stats[n].values()):>11d} " for n in NAMES)
print(row)
print("=" * 86)

# Per-rung attributable delta — the whole point of the ladder.
print(f"\n{'rung':<12}{'variable changed':<26}{'Δ mean pass^k':>16}{'tasks moved':>14}")
print("-" * 68)
for name, _, changed, baseline in LADDER:
    if baseline is None:
        print(f"{name:<12}{'(reference)':<26}{'—':>16}{'—':>14}")
        continue
    before, after = ladder_stats[baseline], ladder_stats[name]
    delta = _suite_mean(after, "pass_hat_k") - _suite_mean(before, "pass_hat_k")
    moved = [t for t in task_order
             if abs(after[t]["pass_hat_k"] - before[t]["pass_hat_k"]) > 1e-9]
    print(f"{name:<12}{changed:<26}{delta:>+15.0%} {len(moved):>13}")
    for t in moved:
        d = after[t]["pass_hat_k"] - before[t]["pass_hat_k"]
        print(f"{'':<38}{t} {before[t]['pass_hat_k']:.0%} → {after[t]['pass_hat_k']:.0%} ({d:+.0%})")
print("-" * 68)

Comparability check (PI003):
  Baseline B  reference run
  V1          comparable to Baseline B (only `system_prompt` differs) ✓
  V2          comparable to V1 (only `tool_specs` differs) ✓
  V3          comparable to V2 (only `tool_implementations` differs) ✓

ABLATION LADDER — pass^5 per task (k=5 trials each, corpus f4af23d76593)
task                    suite         Baseline B          V1          V2          V3
------------------------------------------------------------------------------------
price_jeans             regression         100%        100%        100%        100% 
price_tshirt            capability         100%        100%        100%        100% 
multi_item_total        capability           0%        100%        100%        100% 
percent_off_jacket      capability           0%          0%        100%        100% 
unknown_product_shoes   regression         100%          0%          0%          0% 
catalog_overview        capability           0%          0%        100

In [20]:
# ── Triage: what moved, and what the transcript says about why ────────────────
# A rung that moves a number the wrong way is the most valuable output an eval
# produces, and it is worth nothing without the transcript behind it. For every
# rung this prints: tasks that regressed (with the failing trial), tasks that
# improved, and — just as reportable — rungs where nothing moved at all.

def _tool_calls(trial):
    return [f"{b['name']}({b.get('input')})" for b in trial.get("transcript", [])
            if isinstance(b, dict) and b.get("type") == "tool_use"]


def _worst_trial(run_name, task_id):
    """The first trial of this task that didn't pass — the thing to actually read."""
    for run in ladder_runs[run_name]["runs"]:
        for r in run:
            if r["task_id"] == task_id and not r["passed"]:
                return r
    return None


for _name, _, _changed, _baseline in LADDER:
    if _baseline is None:
        continue
    _before, _after = ladder_stats[_baseline], ladder_stats[_name]
    _lost = [t for t in task_order if _before[t]["pass_hat_k"] > _after[t]["pass_hat_k"]]
    _won  = [t for t in task_order if _before[t]["pass_hat_k"] < _after[t]["pass_hat_k"]]

    print(f"\n{'=' * 86}\n{_name}  —  changed `{_changed}`  (vs {_baseline})\n{'=' * 86}")
    if not _lost and not _won:
        print(f"  Nothing moved. Every task scored identically to {_baseline}.")
        print(f"  That is a result: this corpus contains no task that distinguishes")
        print(f"  `{_changed}` from the configuration before it. Either the change is inert")
        print(f"  for these seven queries, or an earlier rung already covered the failure")
        print(f"  mode it addresses. Reporting it as 'no improvement' would be misleading;")
        print(f"  reporting it as 'untested by this corpus' is the honest version.")
        continue

    for _tid in _won:
        print(f"  ✅ {_tid}: {_before[_tid]['pass_hat_k']:.0%} → {_after[_tid]['pass_hat_k']:.0%} "
              f"({_after[_tid]['passes']}/{_after[_tid]['k']} trials)")

    for _tid in _lost:
        print(f"\n  🔻 REGRESSION — {_tid} ({suite_of[_tid]}): "
              f"{_before[_tid]['pass_hat_k']:.0%} → {_after[_tid]['pass_hat_k']:.0%} "
              f"({_after[_tid]['passes']}/{_after[_tid]['k']} trials passed)")
        _t = _worst_trial(_name, _tid)
        if _t is None:
            print("     (no failing trial recorded — this was an error, not a wrong answer)")
            continue
        print(f"     query        : {_t['query']}")
        print(f"     tool calls   : {_tool_calls(_t) or '(none)'}")
        print(f"     agent said   : {(_t['final_text'] or '(no text)')[:300]}")
        for _g in _t["grades"]:
            if _g["score"] != 1.0:
                _c = _g["check"] if isinstance(_g["check"], str) else json.dumps(_g["check"])
                print(f"     ✗ {_g['type']}: {_c[:110]}")
                print(f"       └─ {_g['reason'][:170]}")
print()


V1  —  changed `system_prompt`  (vs Baseline B)
  ✅ multi_item_total: 0% → 100% (5/5 trials)

  🔻 REGRESSION — unknown_product_shoes (regression): 100% → 0% (0/5 trials passed)
     query        : How much for shoes?
     tool calls   : (none)
     agent said   : I'd be happy to help! We have several types of shoes. Could you be more specific about which shoes you're interested in? For example:

- Sneakers
- Boots
- Flats
- Sandals
- Or something else?

Once you let me know, I can look up the exact price for you.
     ✗ tool_use: {"tool_name": "get_product"}
       └─ 'get_product' never called. Actual: []
     ✗ llm_judge: The response either says the item is unavailable, or offers a specific alternative product from the shop inste
       └─ The response neither states the shoes are unavailable nor offers a specific alternative product; it only asks for clarification.

V2  —  changed `tool_specs`  (vs V1)
  ✅ percent_off_jacket: 0% → 100% (5/5 trials)
  ✅ catalog_overview: 0% → 100% 

---

## Part 7 · Auditing our own harness

Everything above cites Jackdaws. Citing a discipline and *passing* it are different claims, and only one of them is checkable.

`prompt-evaluation-best-practices` ships `scripts/prompt_eval_audit.py` — a stdlib-only auditor for rules PI001–PI014, the measurement-hygiene family: is the corpus frozen, does a run carry a fingerprint, are runs compared only when comparable, does each grader have a negative control, is the judge pinned and its parsing guarded, are truncation and API errors kept out of the quality score. It is advisory by design: it ranks findings and names the reference file that fixes each one, it does not return a verdict.

One wrinkle. The auditor globs `**/*.py` and does not read `.ipynb` — pointed at this folder as-is it reports "no prompt-eval harness in this tree", which is a *true* statement about a directory containing one notebook and no Python. So the cell below exports this notebook to a script into a temp directory first, and audits that. The harness being audited is exactly the code you have been reading.

In [21]:
# ── Run the Jackdaws prompt-eval auditor against this notebook's harness ──────
import os, subprocess, sys, tempfile
from pathlib import Path

def _find_jackdaws():
    """$JACKDAWS_ROOT, else ~/dev/jackdaws, else None — the notebook still runs without it."""
    for candidate in (os.environ.get("JACKDAWS_ROOT"), Path.home() / "dev" / "jackdaws"):
        if not candidate:
            continue
        script = Path(candidate) / "skills/prompt-evaluation-best-practices/scripts/prompt_eval_audit.py"
        if script.is_file():
            return script
    return None


def _find_notebook(name="Building_an_Eval.ipynb"):
    for base in (Path.cwd(), *Path.cwd().parents):
        hit = base / name
        if hit.is_file():
            return hit
        hit = base / "day2" / "01_evals" / name
        if hit.is_file():
            return hit
    return None


AUDIT_SCRIPT = _find_jackdaws()
NOTEBOOK = _find_notebook()

if AUDIT_SCRIPT is None:
    print("Jackdaws not found (set JACKDAWS_ROOT or clone to ~/dev/jackdaws) — skipping the audit.")
    print("Everything above still runs; you just don't get the independent check.")
elif NOTEBOOK is None:
    print("Could not locate Building_an_Eval.ipynb from the current working directory — skipping.")
else:
    with tempfile.TemporaryDirectory() as tmp:
        export = subprocess.run(
            [sys.executable, "-m", "nbconvert", "--to", "script",
             "--output-dir", tmp, str(NOTEBOOK)],
            capture_output=True, text=True,
        )
        exported = sorted(Path(tmp).glob("*.py"))
        if export.returncode != 0 or not exported:
            print("nbconvert --to script failed:\n" + (export.stderr or "").strip()[-1500:])
        else:
            print(f"Auditing: {exported[0].name}  ({exported[0].stat().st_size:,} bytes exported)")
            print(f"Auditor:  {AUDIT_SCRIPT}\n")
            audit = subprocess.run(
                [sys.executable, str(AUDIT_SCRIPT), "--root", tmp, "--format", "text"],
                capture_output=True, text=True,
            )
            print(audit.stdout.strip() or "(no output)")
            if audit.stderr.strip():
                print("\n[stderr]\n" + audit.stderr.strip())
            print(f"\nexit code {audit.returncode}  "
                  f"(0 = clean, 1 = findings — a normal result for an advisory auditor)")

Auditing: Building_an_Eval.py  (116,717 bytes exported)
Auditor:  /Users/jonhaz/dev/jackdaws/skills/prompt-evaluation-best-practices/scripts/prompt_eval_audit.py

not applicable:
  - PI001 (no corpus generator found; the corpus is authored or committed)
  - PI005 (no composite score built from several graders)
  - PI006 (no composite score built from several graders)
  - PI008 (no output-shape criterion is being graded)
  - PI013 (the judge grades against a fixed rubric, not per-case criteria)

summary: 0 finding(s) — 0 error, 0 warning, 0 info; 9/9 applicable controls present, 5 rule(s) not applicable

exit code 0  (0 = clean, 1 = findings — a normal result for an advisory auditor)


### 🧭 Walking the audit

Nine applicable controls, nine present, zero findings. That is only meaningful if each one maps to something you can point at, so here is the mapping — and where the auditor was more generous than it should have been.

| rule | what it demands | where it's closed |
|---|---|---|
| PI002 | a run records a fingerprint | `run_measured` stamps `corpus_sha`, `agent_version`, `agent_model`, `judge_model`, `k`, `max_workers`, `changed`, `baseline_run` into `results["config"]` |
| PI003 | compare only comparable runs | `compare_runs` raises on any difference in `COMPARABLE_KEYS`; the ladder table runs it on every rung before quoting a delta |
| PI004 | one variable per run, recorded | the `LADDER` list — each rung declares its `changed` field, and a run with no declared variable is rejected |
| PI007 | every grader has a negative control it is proven to fail | the negative-controls cell: 12 controls, every grader shown producing both verdicts, `AssertionError` if one can't |
| PI009 | the grader gets no information the prompt was denied | `grade_llm_judge` builds its prompt from `context["query"]`, `result["final_text"]` and one criterion — the catalog and the expected answer are never in scope |
| PI010 | pinned judge, guarded parsing | `JUDGE_MODEL` resolved once and pinned; structured output with a schema enum; `try/except` around both the schema path and the text fallback, and any failure returns 0.0 with the cause in `reason` |
| PI011 | k trials with a spread | `K = 5`, and `passk_table` reports pass@k and pass^k separately rather than a mean that hides the spread |
| PI012 | truncation and API errors distinguished from low quality | `TruncatedRun` raised on `stop_reason == "max_tokens"`, landing in the runner's `error` field; `task_stats` excludes `error` trials from the denominator and counts them separately |
| PI014 | a mandatory violation caps the score | the runner's `passed = all(g["score"] == 1.0 ...)` — grades are a conjunction, never an average, so one failed criterion cannot be diluted by four passing ones |

**Where the auditor was too kind.** It marked **PI013** (per-case criteria frozen and hashed) *not applicable*, on the grounds that "the judge grades against a fixed rubric, not per-case criteria." That detection is conservative and, here, wrong: the criteria *are* per-case — each judged task carries its own criterion strings — and they *are* hashed, because `corpus_fingerprint` covers the whole `graders` block including every criterion. So the control is present; it just isn't present in the shape the auditor scans for. Worth recording as a detection gap in the tool rather than a pass we earned. **PI001** is similar: marked not-applicable because there's no corpus *generator* to catch regenerating between runs, but the corpus is frozen and hashed anyway.

**What the auditor doesn't check, and we didn't build.** PI007 proves a judge can return both verdicts. It does not prove the judge returns the *right* verdict. There is no calibration set here — no batch of responses hand-labelled by a human, scored by the judge, and checked for agreement. Every judged number in this notebook rests on an uncalibrated grader, and that is the single largest unquantified risk in the measurement. `eval-and-testing-best-practices` is explicit that a judge should be calibrated against human labels before its scores are trusted; doing it properly is a bigger job than this session, so it is named here rather than quietly skipped.

---

### 🧭 Decision log 6 — what the numbers actually say

**The headline, and the headline a bundled change would have produced.**

Mean pass^5 across the frozen 7-task corpus went **57% → 57% → 86% → 86%** across the four rungs. Had I done Part 5 the way it's written — edit the agent cell, fix all four defects at once, re-run — the notebook would report **57% → 86%, a 29-point improvement**, and that number would be true. It would also have concealed this:

| suite | Baseline B | V1 | V2 | V3 |
|---|---|---|---|---|
| capability (pass^5) | 25% | 50% | **100%** | 100% |
| regression (pass^5) | **100%** | 67% | 67% | 67% |

The capability suite went from 25% to 100%. The regression suite went from 100% to 67% **and never came back**. A bundled before/after averages those into one rising number. The ladder makes them two separate facts, and the second one is the one you'd want to know before shipping.

**Rung by rung.**

- **V1 — system prompt: net zero, and that "zero" is two large opposite movements.** It fixed `multi_item_total` outright (0% → 100%): telling the agent to route every calculation through `calculate` is exactly what a multi-step total needed. It simultaneously broke `unknown_product_shoes` (100% → 0%, on all five trials). The transcript in the triage cell above is the whole story — asked "How much for shoes?", the V1 agent replies *"We have several types of shoes. Could you be more specific? — Sneakers, Boots, Flats, Sandals"* **without calling `get_product` at all.** It invented a shoe department. The shipped agent passed this task by accident: its lookup crashed with `Error: 'shoes'`, and it reported the failure honestly. My better prompt made the agent more conversational, and a more conversational agent asks a clarifying question instead of checking. The prompt says "if an item is not in the catalog, say so" — it never says *find out first*, and the model has no other way to know.

- **V2 — tool specs: +29 points, the only rung that paid.** Both gains trace to the same line: giving `product` an `enum` of the twelve catalog keys. `catalog_overview` (0% → 100%) becomes answerable because the enum *is* the inventory — before it, the agent had no way to know what the shop sells. `percent_off_jacket` (0% → 100%) is fixed by the `calculate` description saying there is no percentage operator and to express 20% off as a multiplication by 0.8. Both are tool-contract problems, and both were invisible from the outside: the shipped specs described each tool with its own name.

- **V3 — tool implementations: nothing moved, and that is a reportable result.** The normalised lookup and the structured `not_found` payload are, I'd argue, the most correct code in the section. This corpus cannot tell. Worse, and more interesting: **V3's `not_found` result was never once reached**, because V1's regression removed the call site — the agent stopped calling `get_product` for out-of-catalog items entirely. I built a careful error path and then measured a configuration that never executes it. "No improvement" would be the wrong summary. "Untested by this corpus" is the right one.

**What is still broken.**

1. **`unknown_product_shoes` fails on V1, V2 and V3.** It is a *regression*-suite task — a property that must always hold — and it currently holds in none of the improved configurations. On the ladder's own logic this is the thing to fix next, and it outranks every remaining capability gain.
2. **V3 is unmeasured, not validated.** Nothing in the frozen corpus distinguishes `get_product("T-Shirts")` from `get_product("t-shirt")`, or exercises an invalid operator. Those tasks don't exist because the corpus was frozen before V3 was written — and adding them *now*, after seeing the results, is exactly the corpus drift PI001 exists to prevent.
3. **The corpus is close to saturated.** Five of seven tasks sit at 100% by V2. A suite with that little headroom can register a regression but has almost no room left to register an improvement.
4. **The judge is uncalibrated.** It has been proven able to fail (PI007) and never checked for agreement with a human. Every judged number here inherits that.
5. **k = 5 is small.** A task passing 5/5 is still consistent with a true per-trial pass rate as low as ~55% at 95% confidence. `price_jeans` at "100%" means "did not fail five times", not "reliable".

**What I'd do next, in order.**

1. **V4, changing only `system_prompt`, measured against V3.** One added rule: *before saying anything about whether an item is stocked — including before asking a clarifying question — call `get_product`.* That should restore `unknown_product_shoes` and, for the first time, actually route a query through V3's `not_found` path. One variable, one run, attributable.
2. **Corpus v2 — a second frozen suite, hashed separately, never compared to v1 numbers.** Add `price_tshirt_plural` ("how much are T-Shirts?") and `invalid_operator` to give V3 something to be measured by, plus two or three more out-of-catalog phrasings so `unknown_product_shoes` isn't a single point of evidence for a whole regression property.
3. **Calibrate the judge** on ~20 hand-labelled responses spanning both verdicts, and report agreement alongside every judged score.
4. **Raise k on the regression suite only.** Regression tasks are gated on pass^k, which is where small k hurts most; capability tasks can stay at k=5.

**The thing I'd take to a customer.** Not the 29 points. The fact that a change I was confident about — a well-written system prompt replacing "You are a helpful assistant" — silently broke a safety-adjacent behaviour, that no aggregate metric would have shown it, and that the only reason it surfaced is that the run before it was measured separately and the corpus was frozen in between.

---

## Extensions

Finished early? Here are ideas ordered roughly by complexity.

### Quick wins
- **Negative test cases.** Queries the agent *should* handle gracefully: "How much is a piano?" (not in catalog), "What's the meaning of life?" (off-topic).
- **Task validation.** Write a function that checks every task dict has the required fields before running the eval.
- **Generate tasks with Claude.** Ask Claude Code to generate additional eval tasks based on the catalog and the patterns you've established.

### Model comparison
- **Haiku vs Sonnet vs Opus.** Run the same eval across models. Does a more capable model compensate for bad tool specs? Use `run_eval(run_agent, tasks, model="claude-sonnet-5")` and compare results.
- **Cost vs accuracy.** Aggregate token usage and estimated cost per run. Is Sonnet worth the extra cost if Haiku passes 90% of tasks?

### Better graders
- **Efficiency graders.** Grade not just correctness but efficiency: did the agent use the minimum number of tool calls?
- **Grader auto-discovery.** Replace the manual `GRADER_REGISTRY` dict with a `@grader` decorator that registers functions automatically.

### Analysis and visualization
- **Results visualization.** Build a chart (matplotlib, plotly) showing pass rates by category or a comparison across models.
- **Baseline comparison.** Diff two saved JSON results side-by-side, flag regressions.
- **Pass@k and pass^k.** Run each task k times. Pass@k = "passed at least once"; pass^k = "passed every time". Useful for distinguishing flaky from reliably broken.
- **Statistical robustness.** Compute variance, confidence intervals, and statistical significance across runs.

### Harness improvements
- **Max turns guard.** Prevent the agent from looping forever by adding a turn limit.
- **Per-task timeouts.** Kill tasks that take too long.
- **Retry with backoff.** Handle transient API errors gracefully.
- **Caching.** Avoid re-running unchanged tasks by hashing the query + agent config.
- **Progress tracking.** Show a live progress bar as the eval runs.
- **Parameter sweep.** Cartesian product over prompts, effort levels, models, etc.

---

## Beyond This Session

### What we simplified

Some design choices in this eval are intentionally simple. Here's what production systems do differently:

- **Agent instrumentation.** We added an `eval_mode` flag so the agent returns the full transcript. In practice, you don't want to modify the agent at all. Production frameworks wrap the agent and intercept API calls to build the transcript without touching agent code.

- **Observability.** We log the full transcript as a flat list. Production systems use OpenTelemetry span trees to capture structured traces with timing, nesting, and metadata, so you can see not just *what* happened but *how long* each step took and how calls were nested.

- **Environment isolation.** Our agent runs in the same process as the eval. For evals that run code (like SWE-bench), a 3-layer Docker pattern is the standard: (1) base image with OS + runtime, (2) environment image with dependencies installed (content-addressable via hash, shared across tasks), (3) instance image with the specific repo checkout per task.

- **Task format.** We use inline Python dicts. Production harnesses typically use JSONL files (one task per line), YAML with references to external files, or database-backed task stores.

### Existing frameworks

You don't have to build an eval harness from scratch. Several frameworks provide this out of the box:

- **Promptfoo.** Open-source eval framework with a nice CLI and web UI.
- **Braintrust.** Hosted eval platform with logging, tracing, and comparison tools.
- **LangSmith.** Tracing and eval platform from the LangChain ecosystem.
- **Harbor.** Open-source eval framework focused on LLM safety.

Internally, Anthropic uses **eval-toolbox** to add evals to our **eval-registry**. A natural follow-up to this session would be migrating your eval to one of these frameworks.